# Vision Model → Segmentation Taxonomy — Stable Share v1.3.1 AUDITED

**Group-share single-image baseline**

This package preserves the successful v1.3 inference/refinement logic while fixing configuration drift before sharing:

- 30-class taxonomy declared before inference
- `traffic_cone_barrel` formally included as ID 29
- one merged `tree` output class
- strict Google Street View artifact `IGNORE=255`
- fence recovery
- glass-curtain-wall scaffold veto
- final schema / percentage QA

No segmentation thresholds were changed from the successful v1.3 result.


In [ ]:
!pip -q install "transformers>=4.57,<5" "huggingface-hub<1.0" accelerate safetensors pandas matplotlib pillow scikit-learn tqdm opencv-python torchvision
print("✓ Dependencies installed")


In [ ]:
!pip uninstall -y gradio gradio-client -q

print("✓ Unused Gradio packages removed")


In [ ]:
from pathlib import Path
import json, gc, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
from PIL import Image
from IPython.display import display

PROJECT_ROOT = Path("/content/VLM_Street_Interface_Segmentation")
CONFIG_DIR = PROJECT_ROOT / "config"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
for d in [PROJECT_ROOT, CONFIG_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SOURCE_BASELINE_VERSION = "hybrid_v0.10_frozen_baseline-r1"
SHARE_VERSION = "stable-share-v1.3.1-audited"
TAXONOMY_VERSION = "street_interface_v1.3.1"
IGNORE_INDEX = 255

CLASS_NAMES = {0: 'other_unknown',
 1: 'roadway',
 2: 'sidewalk',
 3: 'bike_lane',
 4: 'curb_edge',
 5: 'upper_building_facade',
 6: 'ground_floor_solid_facade',
 7: 'ground_floor_glazing',
 8: 'door_entrance',
 9: 'signboard',
 10: 'awning_canopy',
 11: 'arcade_column',
 12: 'arcade_soffit',
 13: 'sidewalk_shed_scaffold',
 14: 'stoop_stair',
 15: 'wall_ledge',
 16: 'fence_railing',
 17: 'planter_container',
 18: 'tree',
 19: 'shrub_hedge',
 20: 'ground_vegetation',
 21: 'vertical_green_wall',
 22: 'bench_seating',
 23: 'pole_fixture',
 24: 'traffic_sign_signal',
 25: 'person',
 26: 'vehicle',
 27: 'sky',
 28: 'upper_building_glazing',
 29: 'traffic_cone_barrel'}
PALETTE = {0: (0, 0, 0),
 1: (128, 64, 128),
 2: (244, 35, 232),
 3: (255, 100, 100),
 4: (255, 180, 180),
 5: (70, 70, 70),
 6: (110, 80, 70),
 7: (0, 220, 255),
 8: (255, 140, 0),
 9: (255, 220, 0),
 10: (180, 100, 255),
 11: (120, 120, 160),
 12: (170, 170, 210),
 13: (0, 200, 200),
 14: (170, 90, 40),
 15: (150, 120, 80),
 16: (190, 153, 153),
 17: (180, 110, 40),
 18: (50, 130, 20),
 19: (100, 180, 40),
 20: (150, 220, 80),
 21: (0, 150, 80),
 22: (255, 80, 160),
 23: (153, 153, 153),
 24: (255, 255, 0),
 25: (220, 20, 60),
 26: (0, 0, 142),
 27: (70, 130, 180),
 28: (0, 170, 220),
 29: (255, 80, 0)}

LABEL2ID = {name: cid for cid, name in CLASS_NAMES.items()}
ID2LABEL = CLASS_NAMES.copy()
NUM_CLASSES = len(CLASS_NAMES)

# -----------------------------
# Taxonomy integrity invariants
# -----------------------------
assert NUM_CLASSES == 30
assert set(CLASS_NAMES) == set(PALETTE)
assert len(set(CLASS_NAMES.values())) == NUM_CLASSES
assert set(CLASS_NAMES.keys()) == set(range(NUM_CLASSES))
assert LABEL2ID["tree"] == 18
assert LABEL2ID["upper_building_glazing"] == 28
assert LABEL2ID["traffic_cone_barrel"] == 29
assert "tree_canopy" not in LABEL2ID
assert "tree_trunk" not in LABEL2ID
assert IGNORE_INDEX not in CLASS_NAMES

print("Source baseline:", SOURCE_BASELINE_VERSION)
print("Share package:", SHARE_VERSION)
print("Taxonomy:", TAXONOMY_VERSION)
print("Declared classes:", NUM_CLASSES)
print("Tree output policy: leaves + branches + trunk → tree")
print("Traffic-control policy: traffic_cone_barrel is a formal taxonomy class (ID 29)")
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU 未啟用。Colab: Runtime → Change runtime type → T4 GPU.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# AUTHORITATIVE TAXONOMY / ANNOTATION CONFIG — v1.3.1 AUDITED
#
# CLASS_NAMES + PALETTE defined above are the single source of truth.
# Config files are generated from those dictionaries so the runtime
# label map, exported taxonomy, CVAT config, and statistics cannot drift.
# ============================================================

TAXONOMY = {
    "taxonomy_version": TAXONOMY_VERSION,
    "num_classes": NUM_CLASSES,
    "ignore_index": IGNORE_INDEX,
    "classes": [
        {
            "id": cid,
            "name": CLASS_NAMES[cid],
            "rgb": list(PALETTE[cid]),
        }
        for cid in sorted(CLASS_NAMES)
    ],
}

ANNOTATION_RULES = {'other_unknown': {'include': ['Visible regions that cannot be reliably assigned to another taxonomy class after '
                               'class-specific recovery.'],
                   'exclude': ['Google Street View screenshot/UI artifacts; use IGNORE=255.']},
 'roadway': {'include': ['Motor-vehicle travel lanes', 'Visible roadway pavement'],
             'exclude': ['Sidewalk', 'Bike lane when visually identifiable']},
 'sidewalk': {'include': ['Pedestrian walking surface', 'Public sidewalk paving'],
              'exclude': ['Roadway', 'Building steps']},
 'bike_lane': {'include': ['Clearly identifiable bicycle-lane pavement'],
               'exclude': ['Ordinary roadway', 'Ordinary sidewalk']},
 'curb_edge': {'include': ['Visible curb face', 'Raised transition between roadway and pedestrian surface'],
               'exclude': ['Entire sidewalk']},
 'upper_building_facade': {'include': ['Opaque building facade above ground-floor interface zone'],
                           'exclude': ['Upper glazing', 'Ground-floor storefront']},
 'ground_floor_solid_facade': {'include': ['Opaque ground-floor wall or storefront facade'],
                               'exclude': ['Glazing', 'Door', 'Signboard']},
 'ground_floor_glazing': {'include': ['Ground-floor storefront glass',
                                      'Display windows',
                                      'Transparent/reflective ground-floor glazing'],
                          'exclude': ['Upper-floor glazing']},
 'door_entrance': {'include': ['Building entrance door', 'Store entrance', 'Doorway opening'], 'exclude': ['Windows']},
 'signboard': {'include': ['Storefront signage', 'Business fascia/projecting signs'], 'exclude': ['Traffic signs']},
 'awning_canopy': {'include': ['Storefront awning', 'Entrance canopy'],
                   'exclude': ['Construction sidewalk shed', 'Tree']},
 'arcade_column': {'include': ['Columns supporting a building-integrated covered pedestrian arcade'],
                   'exclude': ['Construction scaffold poles', 'Streetlight poles']},
 'arcade_soffit': {'include': ['Underside/ceiling of a building-integrated arcade'],
                   'exclude': ['Construction sidewalk shed']},
 'sidewalk_shed_scaffold': {'include': ['Temporary construction sidewalk shed',
                                        'Scaffolding structure/protective overhead shed'],
                            'exclude': ['Permanent arcade', 'Glass curtain wall', 'Glazed facade']},
 'stoop_stair': {'include': ['Exterior entrance steps', 'Stoop'], 'exclude': ['Ordinary sidewalk']},
 'wall_ledge': {'include': ['Low wall', 'Raised ledge'], 'exclude': ['Fence', 'Planter vegetation']},
 'fence_railing': {'include': ['Park perimeter fence',
                               'Metal fence',
                               'Iron fence',
                               'Pedestrian railing',
                               'Openwork barrier'],
                   'exclude': ['Solid wall', 'Scaffolding', 'Building curtain wall']},
 'planter_container': {'include': ['Built planter box', 'Raised planter', 'Plant container'],
                       'exclude': ['Vegetation inside planter']},
 'tree': {'include': ['Tree leaves', 'Tree crown', 'Visible tree branches', 'Bare branches', 'Main visible tree trunk'],
          'exclude': ['Shrub/hedge', 'Ground vegetation', 'Vertical green wall']},
 'shrub_hedge': {'include': ['Shrub', 'Hedge', 'Dense eye-level woody vegetation'],
                 'exclude': ['Tree', 'Ground grass']},
 'ground_vegetation': {'include': ['Grass', 'Low groundcover vegetation'], 'exclude': ['Shrub', 'Tree']},
 'vertical_green_wall': {'include': ['Vegetation covering a vertical facade/support'],
                         'exclude': ['Tree overlapping wall in perspective']},
 'bench_seating': {'include': ['Bench', 'Fixed public seating'], 'exclude': ['Low wall']},
 'pole_fixture': {'include': ['Streetlight pole', 'Utility pole', 'Fixed vertical street pole'],
                  'exclude': ['Tree trunk', 'Scaffolding pole', 'Google Street View UI artifact']},
 'traffic_sign_signal': {'include': ['Traffic sign', 'Traffic signal'], 'exclude': ['Commercial signboard']},
 'person': {'include': ['Pedestrian', 'Visible person'], 'exclude': []},
 'vehicle': {'include': ['Car', 'Truck', 'Bus', 'Motorcycle', 'Bicycle as vehicle object'], 'exclude': []},
 'sky': {'include': ['Visible sky'], 'exclude': ['Sky reflection in glazing']},
 'upper_building_glazing': {'include': ['Upper-floor windows',
                                        'Curtain-wall glazing',
                                        'Glazed building surfaces above ground-floor interface'],
                            'exclude': ['Ground-floor storefront glazing', 'Vehicle glass']},
 'traffic_cone_barrel': {'include': ['Traffic cone', 'Construction cone', 'Traffic barrel', 'Construction barrel/drum'],
                         'exclude': ['Traffic sign/signal', 'Ordinary street furniture', 'Planter container']}}

BASELINE_SETTINGS = {'packaging_version': 'stable-share-v1.3.1-audited',
 'algorithm_baseline': 'hybrid_v0.10_plus_targeted_v1.3_refinements',
 'baseline_status': 'SHARE_READY_SINGLE_IMAGE',
 'taxonomy_version': 'street_interface_v1.3.1',
 'models': {'full_scene_ade': 'nvidia/segformer-b5-finetuned-ade-640-640',
            'full_scene_mapillary': 'facebook/mask2former-swin-large-mapillary-vistas-semantic',
            'detector': 'IDEA-Research/grounding-dino-base',
            'mask_refiner': 'facebook/sam2.1-hiera-small'},
 'detail_detection_thresholds': {'ground_floor_glazing': 0.2,
                                 'door_entrance': 0.21,
                                 'signboard': 0.24,
                                 'awning_canopy': 0.25,
                                 'sidewalk_shed_scaffold': 0.28,
                                 'stoop_stair': 0.22,
                                 'fence_railing': 0.18,
                                 'planter_container': 0.22,
                                 'bench_seating': 0.24,
                                 'person': 0.27,
                                 'vehicle': 0.28,
                                 'traffic_cone_barrel': 0.18},
 'notes': ['Google Street View UI residuals remain IGNORE=255 and are excluded from measurement statistics.',
           'Fence recovery is class-specific and primarily targets other_unknown pixels.',
           'Glass-dominant vertical building components are vetoed from scaffold.',
           'Tree component prompts are detection-only; taxonomy output is one tree class.',
           'traffic_cone_barrel is a declared taxonomy class from initialization; it is not dynamically appended at '
           'runtime.',
           'other_unknown is retained as an epistemic fallback and is not generically filled.'],
 'declared_num_classes': 30}

CVAT_LABELS = [
    {
        "name": CLASS_NAMES[cid],
        "color": "#%02x%02x%02x" % PALETTE[cid],
        "attributes": [],
    }
    for cid in sorted(CLASS_NAMES)
]

# -----------------------------
# Cross-config audit
# -----------------------------
assert TAXONOMY["num_classes"] == NUM_CLASSES == 30
assert [x["id"] for x in TAXONOMY["classes"]] == list(range(NUM_CLASSES))
assert [x["name"] for x in TAXONOMY["classes"]] == [CLASS_NAMES[i] for i in range(NUM_CLASSES)]
assert [x["name"] for x in CVAT_LABELS] == [CLASS_NAMES[i] for i in range(NUM_CLASSES)]
assert set(ANNOTATION_RULES) == set(CLASS_NAMES.values())
assert "tree_canopy" not in ANNOTATION_RULES
assert "tree_trunk" not in ANNOTATION_RULES

for name, obj in [
    ("taxonomy_v1_3_1.json", TAXONOMY),
    ("annotation_rules_v1_3_1.json", ANNOTATION_RULES),
    ("baseline_settings_v1_3_1.json", BASELINE_SETTINGS),
    ("cvat_labels_v1_3_1.json", CVAT_LABELS),
]:
    with open(CONFIG_DIR / name, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

print("✓ v1.3.1 audited config exported to", CONFIG_DIR)
print("✓ Runtime taxonomy = exported taxonomy = CVAT taxonomy")
print("✓ 30 declared classes, including traffic_cone_barrel")
print("✓ Tree taxonomy is one class: tree")


## Input
Upload exactly one street-view image. The filename stem becomes the test ID.


In [ ]:
from google.colab import files
import io

print("Upload exactly ONE street-view image for this run.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError(f"Expected exactly 1 image, received {len(uploaded)}")

filename = next(iter(uploaded.keys()))
image = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")
W, H = image.size

# Stable test ID: filename stem by default.
TEST_ID = re.sub(r"[^A-Za-z0-9_-]+", "_", Path(filename).stem).upper()

print("Current test:", TEST_ID)
print("Image:", filename)
print("Size:", image.size)
display(image)


## Runtime helpers


In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc
import cv2

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

assert "upper_building_glazing" in LABEL2ID

W, H = image.size

print("Device:", device)
print("Classes:", len(CLASS_NAMES))
print("Image:", W, "x", H)

def render_taxonomy(label_map):

    rgb = np.zeros(
        (
            label_map.shape[0],
            label_map.shape[1],
            3
        ),
        dtype=np.uint8
    )

    for class_id, color in PALETTE.items():

        rgb[
            label_map == class_id
        ] = color

    return rgb


UNKNOWN_ID = LABEL2ID["other_unknown"]

print("✓ renderer ready")

for name in [
    "ade_model",
    "map_model",
    "grounding_model",
    "sam_model_new",
    "tree_detector",
    "tree_sam"
]:
    try:
        del globals()[name]
    except:
        pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✓ GPU cleared")


## ADE20K full-scene pass


In [ ]:
import torch.nn.functional as F

from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation
)

ADE_MODEL_ID = (
    "nvidia/segformer-b5-finetuned-ade-640-640"
)

print("Loading ADE20K...")

ade_processor = (
    SegformerImageProcessor
    .from_pretrained(
        ADE_MODEL_ID
    )
)

ade_model = (
    SegformerForSemanticSegmentation
    .from_pretrained(
        ADE_MODEL_ID
    )
    .to(device)
)

ade_model.eval()

print("✓ ADE loaded")

@torch.no_grad()
def run_ade(image):

    W, H = image.size

    inputs = ade_processor(
        images=image,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    outputs = ade_model(**inputs)

    logits = F.interpolate(
        outputs.logits,
        size=(H, W),
        mode="bilinear",
        align_corners=False
    )

    probs = torch.softmax(
        logits,
        dim=1
    )

    confidence, prediction = probs.max(
        dim=1
    )

    return (
        prediction[0]
        .cpu()
        .numpy()
        .astype(np.int32),

        confidence[0]
        .cpu()
        .numpy()
        .astype(np.float32)
    )


ade_pred, ade_confidence = run_ade(
    image
)

ADE_ID2LABEL = {
    int(k): str(v).lower()
    for k, v
    in ade_model.config.id2label.items()
}

print("✓ ADE inference complete")

def ade_ids_fuzzy(
    include_terms,
    exclude_terms=None
):

    if exclude_terms is None:
        exclude_terms = []

    found = []

    for class_id, name in ADE_ID2LABEL.items():

        included = any(
            term.lower() in name
            for term in include_terms
        )

        excluded = any(
            term.lower() in name
            for term in exclude_terms
        )

        if included and not excluded:
            found.append(class_id)

    return found


def ade_mask_fuzzy(
    include_terms,
    exclude_terms=None
):

    ids = ade_ids_fuzzy(
        include_terms,
        exclude_terms
    )

    if not ids:

        return np.zeros(
            (H, W),
            dtype=bool
        )

    return np.isin(
        ade_pred,
        ids
    )


ADE_GLASS_V2 = ade_mask_fuzzy([
    "window",
    "glass"
])

ADE_ROAD_V2 = ade_mask_fuzzy([
    "road"
])

ADE_SIDEWALK_V2 = ade_mask_fuzzy([
    "sidewalk",
    "path"
])

ADE_BUILDING_V2 = ade_mask_fuzzy([
    "building",
    "house",
    "skyscraper"
])

ADE_SKY_V2 = ade_mask_fuzzy([
    "sky"
])

ADE_TREE_V2 = ade_mask_fuzzy([
    "tree"
])

ADE_PLANT_V2 = ade_mask_fuzzy([
    "plant",
    "grass",
    "flower"
])

ADE_FENCE_V3 = ade_mask_fuzzy([
    "fence",
    "railing"
])


print("ADE glass:", int(ADE_GLASS_V2.sum()))
print("ADE building:", int(ADE_BUILDING_V2.sum()))
print("ADE trees:", int(ADE_TREE_V2.sum()))
print("ADE fence/railing:", int(ADE_FENCE_V3.sum()))

del ade_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✓ ADE model released")


## Mapillary full-scene pass


In [ ]:
from transformers import (
    AutoImageProcessor,
    Mask2FormerForUniversalSegmentation
)

MAPILLARY_MODEL_ID = (
    "facebook/"
    "mask2former-swin-large-"
    "mapillary-vistas-semantic"
)

print("Loading Mapillary...")

map_processor = (
    AutoImageProcessor
    .from_pretrained(
        MAPILLARY_MODEL_ID
    )
)

map_model = (
    Mask2FormerForUniversalSegmentation
    .from_pretrained(
        MAPILLARY_MODEL_ID
    )
    .to(device)
)

map_model.eval()

print("✓ Mapillary loaded")

inputs = map_processor(
    images=image,
    return_tensors="pt"
)

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

with torch.no_grad():

    outputs = map_model(
        **inputs
    )


map_pred = (
    map_processor
    .post_process_semantic_segmentation(
        outputs,
        target_sizes=[
            (H, W)
        ]
    )[0]
    .cpu()
    .numpy()
    .astype(np.int32)
)


MAP_ID2LABEL = {
    int(k): str(v).lower()
    for k, v
    in map_model.config.id2label.items()
}

print("✓ Mapillary inference complete")

def mapillary_ids(
    include_terms,
    exclude_terms=None
):

    if exclude_terms is None:
        exclude_terms = []

    found = []

    for class_id, name in MAP_ID2LABEL.items():

        included = any(
            term.lower() in name
            for term in include_terms
        )

        excluded = any(
            term.lower() in name
            for term in exclude_terms
        )

        if included and not excluded:
            found.append(class_id)

    return found


def mapillary_mask(
    include_terms,
    exclude_terms=None
):

    ids = mapillary_ids(
        include_terms,
        exclude_terms
    )

    if not ids:

        return np.zeros(
            (H, W),
            dtype=bool
        )

    return np.isin(
        map_pred,
        ids
    )


MAP_ROAD = mapillary_mask(
    ["road"],
    [
        "marking",
        "bike lane",
        "service lane"
    ]
)

MAP_ROAD_MARKINGS = mapillary_mask(
    ["marking"]
)

MAP_SIDEWALK = mapillary_mask(
    ["sidewalk"]
)

MAP_BIKE_LANE = mapillary_mask(
    ["bike lane"]
)

MAP_CURB = mapillary_mask(
    ["curb"]
)

MAP_BUILDING = mapillary_mask(
    ["building"]
)

MAP_SKY = mapillary_mask(
    ["sky"]
)

MAP_FENCE = mapillary_mask(
    ["fence"]
)

MAP_WINDOW = mapillary_mask(
    ["window", "glass"]
)


print("Road:", int(MAP_ROAD.sum()))
print("Sidewalk:", int(MAP_SIDEWALK.sum()))
print("Bike lane:", int(MAP_BIKE_LANE.sum()))
print("Building:", int(MAP_BUILDING.sum()))
print("Sky:", int(MAP_SKY.sum()))
print("Fence:", int(MAP_FENCE.sum()))
print("Window/glass:", int(MAP_WINDOW.sum()))

final_test = np.full(
    (H, W),
    UNKNOWN_ID,
    dtype=np.uint8
)


# Road
final_test[
    MAP_ROAD |
    MAP_ROAD_MARKINGS
] = LABEL2ID["roadway"]


# Bike lane overrides road
final_test[
    MAP_BIKE_LANE
] = LABEL2ID["bike_lane"]


# Sidewalk
final_test[
    MAP_SIDEWALK
] = LABEL2ID["sidewalk"]


# Curb
final_test[
    MAP_CURB
] = LABEL2ID["curb_edge"]


# Building
final_test[
    MAP_BUILDING
] = LABEL2ID[
    "upper_building_facade"
]


# Sky
final_test[
    MAP_SKY
] = LABEL2ID["sky"]


# ADE fallback
unknown = (
    final_test == UNKNOWN_ID
)

final_test[
    unknown & ADE_ROAD_V2
] = LABEL2ID["roadway"]


unknown = (
    final_test == UNKNOWN_ID
)

final_test[
    unknown & ADE_SIDEWALK_V2
] = LABEL2ID["sidewalk"]


unknown = (
    final_test == UNKNOWN_ID
)

final_test[
    unknown & ADE_BUILDING_V2
] = LABEL2ID[
    "upper_building_facade"
]


unknown = (
    final_test == UNKNOWN_ID
)

final_test[
    unknown & ADE_SKY_V2
] = LABEL2ID["sky"]


# preliminary vegetation
final_test[
    ADE_TREE_V2
] = LABEL2ID[
    "tree"
]


building_context = (
    MAP_BUILDING
    |
    ADE_BUILDING_V2
)


# ADE preliminary glazing
basic_glass = (
    ADE_GLASS_V2
    &
    building_context
)

final_test[
    basic_glass
] = LABEL2ID[
    "upper_building_glazing"
]


print("✓ Full scene base created")

rgb_base = render_taxonomy(
    final_test
)

plt.figure(
    figsize=(18, 8)
)

plt.subplot(1, 2, 1)

plt.imshow(image)

plt.title(
    f"{TEST_ID} — Original"
)

plt.axis("off")


plt.subplot(1, 2, 2)

plt.imshow(
    rgb_base
)

plt.title(
    f"{TEST_ID} — Full Scene Base"
)

plt.axis("off")


plt.tight_layout()
plt.show()

del map_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✓ Mapillary model released")


## Grounding DINO detections


In [ ]:
from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection
)

GROUNDING_MODEL_ID = (
    "IDEA-Research/grounding-dino-base"
)

grounding_processor = (
    AutoProcessor.from_pretrained(
        GROUNDING_MODEL_ID
    )
)

grounding_model = (
    AutoModelForZeroShotObjectDetection
    .from_pretrained(
        GROUNDING_MODEL_ID
    )
    .to(device)
)

grounding_model.eval()

print("✓ Grounding DINO loaded")

AUTO_LABEL_QUERIES = {

    "ground_floor_glazing": [
        "storefront glass window",
        "shopfront window",
        "ground floor glass window",
        "glass storefront"
    ],

    "door_entrance": [
        "building entrance door",
        "store entrance door",
        "entrance doorway"
    ],

    "signboard": [
        "storefront business sign",
        "shop signboard",
        "commercial sign"
    ],

    "awning_canopy": [
        "storefront awning",
        "entrance awning"
    ],

    "sidewalk_shed_scaffold": [
        "construction sidewalk shed",
        "sidewalk scaffolding",
        "construction scaffolding",
        "scaffold structure"
    ],

    "stoop_stair": [
        "building entrance steps",
        "residential stoop",
        "entrance stairs"
    ],

    "fence_railing": [
        "park perimeter fence",
        "black iron park fence",
        "metal park fence",
        "pedestrian railing",
        "metal railing",
        "ornamental iron fence",
        "openwork metal fence"
    ],

    "planter_container": [
        "street planter",
        "planter box",
        "plant container"
    ],

    "bench_seating": [
        "street bench",
        "outdoor bench"
    ],

    "person": [
        "pedestrian",
        "person"
    ],

    "vehicle": [
        "car",
        "truck",
        "bus",
        "bicycle"
    ]
}


CLASS_THRESHOLDS = {

    "ground_floor_glazing": 0.20,
    "door_entrance": 0.21,
    "signboard": 0.24,
    "awning_canopy": 0.25,

    "sidewalk_shed_scaffold": 0.28,

    "stoop_stair": 0.22,
    "fence_railing": 0.18,
    "planter_container": 0.22,
    "bench_seating": 0.24,

    "person": 0.27,
    "vehicle": 0.28
}

print("✓ frozen detail configuration loaded")

DETAIL_CROPS = {

    "left_interface": (
        0,
        int(H * 0.22),
        int(W * 0.58),
        int(H * 0.88)
    ),

    "center_interface": (
        int(W * 0.18),
        int(H * 0.20),
        int(W * 0.82),
        int(H * 0.90)
    ),

    "right_interface": (
        int(W * 0.40),
        int(H * 0.18),
        W,
        int(H * 0.90)
    )
}

@torch.no_grad()
def detect_one_class(
    source_image,
    class_name,
    offset_x=0,
    offset_y=0
):

    phrases = AUTO_LABEL_QUERIES[
        class_name
    ]

    inputs = grounding_processor(
        images=source_image,
        text=[phrases],
        return_tensors="pt"
    ).to(device)

    outputs = grounding_model(
        **inputs
    )

    results = (
        grounding_processor
        .post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold=CLASS_THRESHOLDS[
                class_name
            ],
            text_threshold=0.18,
            target_sizes=[
                (
                    source_image.height,
                    source_image.width
                )
            ]
        )
    )[0]

    detections = []

    for box, score in zip(
        results["boxes"],
        results["scores"]
    ):

        x1, y1, x2, y2 = (
            box.detach()
            .cpu()
            .tolist()
        )

        detections.append({

            "class_name":
                class_name,

            "class_id":
                LABEL2ID[
                    class_name
                ],

            "score":
                float(
                    score.detach()
                    .cpu()
                    .item()
                ),

            "box": [
                x1 + offset_x,
                y1 + offset_y,
                x2 + offset_x,
                y2 + offset_y
            ]
        })

    return detections

DETAIL_CLASSES = [
    "ground_floor_glazing",
    "door_entrance",
    "signboard",
    "awning_canopy",
    "sidewalk_shed_scaffold",
    "stoop_stair",
    "fence_railing",
    "planter_container",
    "bench_seating",
    "person",
    "vehicle"
]


detail_detections_raw = []


for class_name in DETAIL_CLASSES:

    print(
        "Detail:",
        class_name
    )

    for crop_name, crop_box in (
        DETAIL_CROPS.items()
    ):

        x1, y1, x2, y2 = crop_box

        crop = image.crop(
            crop_box
        )

        detections = detect_one_class(
            crop,
            class_name,
            offset_x=x1,
            offset_y=y1
        )

        detail_detections_raw.extend(
            detections
        )


print(
    "Raw detail detections:",
    len(detail_detections_raw)
)

WINDOW_PROMPTS = [
    "apartment building window",
    "building window",
    "residential window",
    "glass window",
    "storefront window",
    "glass curtain wall"
]


FACADE_TILES = {

    "left_upper": (
        0,
        0,
        int(W * 0.60),
        int(H * 0.68)
    ),

    "center_upper": (
        int(W * 0.15),
        0,
        int(W * 0.85),
        int(H * 0.72)
    ),

    "right_upper": (
        int(W * 0.40),
        0,
        W,
        int(H * 0.70)
    )
}

@torch.no_grad()
def detect_windows(
    crop_image,
    offset_x,
    offset_y
):

    inputs = grounding_processor(
        images=crop_image,
        text=[WINDOW_PROMPTS],
        return_tensors="pt"
    ).to(device)

    outputs = grounding_model(
        **inputs
    )

    results = (
        grounding_processor
        .post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold=0.15,
            text_threshold=0.17,
            target_sizes=[
                (
                    crop_image.height,
                    crop_image.width
                )
            ]
        )
    )[0]

    detections = []

    for box, score in zip(
        results["boxes"],
        results["scores"]
    ):

        x1, y1, x2, y2 = (
            box.detach()
            .cpu()
            .tolist()
        )

        detections.append({

            "class_name":
                "building_glazing",

            "score":
                float(
                    score.detach()
                    .cpu()
                    .item()
                ),

            "box": [
                x1 + offset_x,
                y1 + offset_y,
                x2 + offset_x,
                y2 + offset_y
            ]
        })

    return detections

window_detections_raw = []

for tile_name, crop_box in (
    FACADE_TILES.items()
):

    print(
        "Window:",
        tile_name
    )

    x1, y1, x2, y2 = crop_box

    crop = image.crop(
        crop_box
    )

    window_detections_raw.extend(
        detect_windows(
            crop,
            x1,
            y1
        )
    )


print(
    "Raw window detections:",
    len(window_detections_raw)
)

TREE_QUERY_GROUPS = {

    "foliage": [
        "green tree foliage",
        "leafy tree crown",
        "tree leaves",
        "green leaves on a tree"
    ],

    "woody": [
        "leafless tree",
        "bare tree branches",
        "tree branches",
        "tree component",
        "bare tree component"
    ]
}


TREE_ROIS = {

    "left_tree": (
        0,
        0,
        int(W * 0.52),
        int(H * 0.78)
    ),

    "center_tree": (
        int(W * 0.18),
        0,
        int(W * 0.75),
        int(H * 0.75)
    ),

    "right_tree": (
        int(W * 0.45),
        0,
        W,
        int(H * 0.75)
    )
}

@torch.no_grad()
def detect_tree_component(
    crop_image,
    prompts,
    component_name,
    offset_x,
    offset_y,
    threshold
):

    inputs = grounding_processor(
        images=crop_image,
        text=[prompts],
        return_tensors="pt"
    ).to(device)

    outputs = grounding_model(
        **inputs
    )

    results = (
        grounding_processor
        .post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold=threshold,
            text_threshold=0.17,
            target_sizes=[
                (
                    crop_image.height,
                    crop_image.width
                )
            ]
        )
    )[0]

    detections = []

    for box, score in zip(
        results["boxes"],
        results["scores"]
    ):

        x1, y1, x2, y2 = (
            box.detach()
            .cpu()
            .tolist()
        )

        detections.append({

            "component":
                component_name,

            "score":
                float(
                    score.detach()
                    .cpu()
                    .item()
                ),

            "box": [
                x1 + offset_x,
                y1 + offset_y,
                x2 + offset_x,
                y2 + offset_y
            ]
        })

    return detections

tree_detections_raw = []

for roi_name, roi_box in TREE_ROIS.items():

    x1, y1, x2, y2 = roi_box

    crop = image.crop(
        roi_box
    )

    tree_detections_raw.extend(

        detect_tree_component(
            crop,
            TREE_QUERY_GROUPS["foliage"],
            "foliage",
            x1,
            y1,
            threshold=0.27
        )
    )

    tree_detections_raw.extend(

        detect_tree_component(
            crop,
            TREE_QUERY_GROUPS["woody"],
            "woody",
            x1,
            y1,
            threshold=0.18
        )
    )


print(
    "Raw tree detections:",
    len(tree_detections_raw)
)

from torchvision.ops import nms


def nms_group(
    detections,
    key_name,
    iou=0.42
):

    if not detections:
        return []

    result = []

    groups = sorted(
        set(
            d[key_name]
            for d in detections
        )
    )

    for group in groups:

        subset = [
            d
            for d in detections
            if d[key_name] == group
        ]

        boxes = torch.tensor(
            [
                d["box"]
                for d in subset
            ],
            dtype=torch.float32
        )

        scores = torch.tensor(
            [
                d["score"]
                for d in subset
            ],
            dtype=torch.float32
        )

        keep = nms(
            boxes,
            scores,
            iou
        )

        result.extend(
            subset[i]
            for i in keep.tolist()
        )

    return result


detail_detections = nms_group(
    detail_detections_raw,
    "class_name"
)

window_detections = nms_group(
    window_detections_raw,
    "class_name"
)

tree_detections = nms_group(
    tree_detections_raw,
    "component"
)


print(
    "Detail:",
    len(detail_detections)
)

print(
    "Windows:",
    len(window_detections)
)

print(
    "Trees:",
    len(tree_detections)
)

del grounding_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✓ DINO released")


## SAM2 mask generation


In [ ]:
from transformers import (
    Sam2Processor,
    Sam2Model
)

SAM_MODEL_ID = (
    "facebook/sam2.1-hiera-small"
)

sam_processor = (
    Sam2Processor.from_pretrained(
        SAM_MODEL_ID
    )
)

sam_model = (
    Sam2Model.from_pretrained(
        SAM_MODEL_ID
    )
    .to(device)
)

sam_model.eval()

print("✓ SAM2 loaded")

@torch.no_grad()
def boxes_to_masks(
    image,
    detections
):

    if not detections:
        return []

    boxes = [
        d["box"]
        for d in detections
    ]

    inputs = (
        sam_processor(
            images=image,
            input_boxes=[boxes],
            return_tensors="pt"
        )
        .to(device)
    )

    outputs = sam_model(
        **inputs,
        multimask_output=False
    )

    masks = (
        sam_processor
        .post_process_masks(
            outputs.pred_masks.cpu(),
            inputs["original_sizes"]
        )[0]
    )

    output = []

    for i, det in enumerate(
        detections
    ):

        mask = masks[i]

        while mask.ndim > 2:
            mask = mask[0]

        output.append({
            **det,
            "mask":
                mask.numpy() > 0
        })

    return output

detail_objects = boxes_to_masks(
    image,
    detail_detections
)

window_objects = boxes_to_masks(
    image,
    window_detections
)

tree_objects = boxes_to_masks(
    image,
    tree_detections
)


print(
    "Detail masks:",
    len(detail_objects)
)

print(
    "Window masks:",
    len(window_objects)
)

print(
    "Tree masks:",
    len(tree_objects)
)


## Core hybrid fusion


In [ ]:
final_v06 = final_test.copy()


INTERFACE_ANCHOR_CLASSES = {
    "ground_floor_glazing",
    "door_entrance",
    "signboard",
    "awning_canopy",
    "sidewalk_shed_scaffold",
    "stoop_stair"
}


interface_anchor = np.zeros(
    (H, W),
    dtype=bool
)


for obj in detail_objects:

    if (
        obj["class_name"]
        in INTERFACE_ANCHOR_CLASSES
        and
        obj["score"] >= 0.23
    ):

        interface_anchor |= obj[
            "mask"
        ]


kernel = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (51, 31)
)


interface_neighborhood = (
    cv2.dilate(
        interface_anchor.astype(
            np.uint8
        ),
        kernel,
        iterations=1
    ) > 0
)


ground_floor_solid = (
    building_context
    &
    interface_neighborhood
    &
    ~ADE_GLASS_V2
)


final_v06[
    ground_floor_solid
] = LABEL2ID[
    "ground_floor_solid_facade"
]


print("✓ v0.6 interface neighborhood built")

facade_glass_aggressive = np.zeros(
    (H, W),
    dtype=bool
)


for obj in window_objects:

    gated = (
        obj["mask"]
        &
        building_context
    )

    original_area = max(
        int(
            obj["mask"].sum()
        ),
        1
    )

    agreement = (
        gated.sum()
        /
        original_area
    )

    if agreement < 0.25:
        continue

    facade_glass_aggressive |= gated


ground_glass = (
    facade_glass_aggressive
    &
    interface_neighborhood
)

upper_glass = (
    facade_glass_aggressive
    &
    ~interface_neighborhood
)


final_v06[
    ground_glass
] = LABEL2ID[
    "ground_floor_glazing"
]

final_v06[
    upper_glass
] = LABEL2ID[
    "upper_building_glazing"
]


print(
    "Aggressive upper glass:",
    upper_glass.mean() * 100
)

FUSION_MIN_SCORE = {

    "ground_floor_glazing":
        0.22,

    "door_entrance":
        0.23,

    "signboard":
        0.26,

    "awning_canopy":
        0.27,

    "sidewalk_shed_scaffold":
        0.28,

    "stoop_stair":
        0.24,

    "fence_railing":
        0.20,

    "planter_container":
        0.24,

    "bench_seating":
        0.25,

    "person":
        0.27,

    "vehicle":
        0.28
}


for obj in detail_objects:

    class_name = obj[
        "class_name"
    ]

    score = obj[
        "score"
    ]

    if (
        score
        <
        FUSION_MIN_SCORE[
            class_name
        ]
    ):
        continue

    mask = obj[
        "mask"
    ]


    if (
        class_name
        ==
        "ground_floor_glazing"
    ):

        allowed = (
            building_context
            &
            interface_neighborhood
        )


    elif class_name in {
        "door_entrance",
        "signboard",
        "awning_canopy"
    }:

        allowed = (
            building_context
            |
            interface_neighborhood
        )


    elif (
        class_name
        ==
        "sidewalk_shed_scaffold"
    ):

        allowed = (
            MAP_BUILDING
            |
            MAP_SIDEWALK
        )


    elif class_name == "fence_railing":

        # v1.3: park perimeter fences are often next to open space,
        # not directly on a building footprint.
        fence_sidewalk_near = cv2.dilate(
            MAP_SIDEWALK.astype(np.uint8),
            cv2.getStructuringElement(cv2.MORPH_RECT, (81, 51)),
            iterations=1
        ) > 0

        allowed = (
            MAP_FENCE
            | ADE_FENCE_V3
            | fence_sidewalk_near
            | building_context
        )

    elif class_name in {
        "stoop_stair",
        "planter_container",
        "bench_seating"
    }:

        allowed = (
            MAP_SIDEWALK
            |
            building_context
        )


    else:

        allowed = np.ones(
            (H, W),
            dtype=bool
        )


    gated = (
        mask
        &
        allowed
    )


    original_area = max(
        int(mask.sum()),
        1
    )


    if (
        gated.sum()
        /
        original_area
        <
        0.12
    ):
        continue


    final_v06[
        gated
    ] = LABEL2ID[
        class_name
    ]


print("✓ v0.6-style detail fusion complete")

dynamic_mask = np.isin(
    final_v06,
    [
        LABEL2ID["person"],
        LABEL2ID["vehicle"]
    ]
)


final_v06[
    MAP_ROAD
    &
    ~dynamic_mask
] = LABEL2ID[
    "roadway"
]


final_v06[
    MAP_BIKE_LANE
    &
    ~dynamic_mask
] = LABEL2ID[
    "bike_lane"
]


final_v06[
    MAP_CURB
    &
    ~dynamic_mask
] = LABEL2ID[
    "curb_edge"
]


print("✓ v0.6 street surfaces protected")

def mask_geometry(mask):

    ys, xs = np.where(mask)

    if len(xs) == 0:

        return {
            "area_ratio": 0,
            "cx": 0,
            "cy": 0
        }

    return {

        "area_ratio":
            float(mask.mean()),

        "cx":
            float(xs.mean() / W),

        "cy":
            float(ys.mean() / H)
    }


accepted_windows = []


for obj in window_objects:

    geometry = mask_geometry(
        obj["mask"]
    )

    # Reject huge window hallucinations
    if (
        geometry["area_ratio"]
        >
        0.015
    ):
        continue

    if (
        geometry["cy"]
        >
        0.80
    ):
        continue

    gated = (
        obj["mask"]
        &
        building_context
    )

    original_area = max(
        int(
            obj["mask"].sum()
        ),
        1
    )

    agreement = (
        gated.sum()
        /
        original_area
    )

    if agreement < 0.50:
        continue

    accepted_windows.append({
        **obj,
        "mask":
            gated
    })


print(
    "Windows:",
    len(window_objects),
    "→ accepted:",
    len(accepted_windows)
)

final_v09 = final_v06.copy()


UPPER_FACADE_ID = LABEL2ID[
    "upper_building_facade"
]

UPPER_GLASS_ID = LABEL2ID[
    "upper_building_glazing"
]


precise_window_mask = np.zeros(
    (H, W),
    dtype=bool
)


for obj in accepted_windows:

    precise_window_mask |= (
        obj["mask"]
    )

SCAFFOLD_ID = LABEL2ID[
    "sidewalk_shed_scaffold"
]


scaffold_core = (
    final_v06
    ==
    SCAFFOLD_ID
)


def shift_mask(mask, dy=0, dx=0):

    out = np.zeros_like(
        mask,
        dtype=bool
    )

    y0 = max(0, -dy)
    y1 = mask.shape[0] - max(0, dy)

    x0 = max(0, -dx)
    x1 = mask.shape[1] - max(0, dx)

    yd0 = max(0, dy)
    xd0 = max(0, dx)

    if y1 > y0 and x1 > x0:

        out[
            yd0:yd0 + (y1-y0),
            xd0:xd0 + (x1-x0)
        ] = mask[
            y0:y1,
            x0:x1
        ]

    return out


under_scaffold_keep = np.zeros(
    (H, W),
    dtype=bool
)


for dy in [
    0,
    20,
    40,
    60,
    80,
    100
]:

    under_scaffold_keep |= (
        shift_mask(
            scaffold_core,
            dy=dy
        )
    )


under_scaffold_keep = (
    cv2.dilate(
        under_scaffold_keep.astype(
            np.uint8
        ),
        cv2.getStructuringElement(
            cv2.MORPH_RECT,
            (71, 29)
        ),
        iterations=1
    ) > 0
)


print(
    "Protected under-scaffold:",
    int(
        under_scaffold_keep.sum()
    )
)

upper_precision_zone = np.zeros(
    (H, W),
    dtype=bool
)

upper_precision_zone[
    :int(H * 0.55),
    :
] = True


upper_precision_zone &= (
    building_context
)

upper_precision_zone &= (
    ~under_scaffold_keep
)


# Reset building pixels in upper precision zone
# to solid façade first
final_v09[
    upper_precision_zone
] = UPPER_FACADE_ID


# Then install only accepted windows
final_v09[
    precise_window_mask
    &
    upper_precision_zone
] = UPPER_GLASS_ID


print(
    "✓ upper building precision transplanted"
)

STOOP_ID = LABEL2ID[
    "stoop_stair"
]


SAFE_ANCHOR_CLASSES = {
    "ground_floor_glazing",
    "door_entrance",
    "signboard"
}


safe_anchor = np.zeros(
    (H, W),
    dtype=bool
)


for obj in detail_objects:

    if (
        obj["class_name"]
        not in
        SAFE_ANCHOR_CLASSES
    ):
        continue

    if obj["score"] < 0.25:
        continue

    if (
        mask_geometry(
            obj["mask"]
        )["area_ratio"]
        >
        0.04
    ):
        continue

    safe_anchor |= (
        obj["mask"]
    )


safe_neighborhood = (
    cv2.dilate(
        safe_anchor.astype(
            np.uint8
        ),
        cv2.getStructuringElement(
            cv2.MORPH_RECT,
            (31, 21)
        ),
        iterations=1
    ) > 0
)


invalid_stoop = (
    (final_v09 == STOOP_ID)
    &
    ~(
        MAP_SIDEWALK
        |
        safe_neighborhood
    )
)


final_v09[
    invalid_stoop
] = LABEL2ID[
    "ground_floor_solid_facade"
]


print(
    "Removed invalid stoop:",
    int(
        invalid_stoop.sum()
    )
)

foliage_mask = np.zeros(
    (H, W),
    dtype=bool
)

woody_mask = np.zeros(
    (H, W),
    dtype=bool
)


for obj in tree_objects:

    mask = obj[
        "mask"
    ]

    valid = (
        mask
        &
        ~MAP_ROAD
        &
        ~MAP_BIKE_LANE
        &
        ~MAP_SKY
    )


    if (
        obj["component"]
        ==
        "foliage"
        and
        obj["score"] >= 0.27
    ):

        foliage_mask |= valid


    elif (
        obj["component"]
        ==
        "woody"
        and
        obj["score"] >= 0.18
    ):

        woody_mask |= valid


# foliage has priority
woody_mask &= (
    ~foliage_mask
)


print(
    "Foliage:",
    foliage_mask.mean() * 100,
    "%"
)

print(
    "Woody:",
    woody_mask.mean() * 100,
    "%"
)

TREE_ID = LABEL2ID[
    "tree"
]

TRUNK_ID = LABEL2ID[
    "tree"
]


protected_interface = np.isin(
    final_v09,
    [
        LABEL2ID["signboard"],
        LABEL2ID["door_entrance"],
        LABEL2ID[
            "sidewalk_shed_scaffold"
        ],
        LABEL2ID["awning_canopy"],
        LABEL2ID["person"],
        LABEL2ID["vehicle"]
    ]
)


foliage_mask &= (
    ~protected_interface
)

woody_mask &= (
    ~protected_interface
)


print(
    "✓ tree masks gated"
)

final_v010 = (
    final_v09.copy()
)


old_tree = (
    (final_v010 == TREE_ID)
    |
    (final_v010 == TRUNK_ID)
)


# Remove previous coarse tree classification
final_v010[
    old_tree
    &
    building_context
] = UPPER_FACADE_ID


final_v010[
    old_tree
    &
    MAP_SIDEWALK
] = LABEL2ID[
    "sidewalk"
]


# Dedicated tree refinement
final_v010[
    woody_mask
] = TRUNK_ID


final_v010[
    foliage_mask
] = TREE_ID


print(
    "✓ v0.10 candidate complete"
)

# ------------------------------------------------------------
# Legacy broad corner watermark masking is intentionally disabled.
#
# It used to force the lower left/right corners to UNKNOWN by
# geometry alone. v1.1 instead detects the actual Street View
# screenshot artifacts later and restores their underlying
# semantic labels from context.
# ------------------------------------------------------------

yy_grid, xx_grid = np.indices((H, W))

legacy_watermark_region = (
    (yy_grid > H * 0.94)
    &
    ((xx_grid < W * 0.20) | (xx_grid > W * 0.80))
)

watermark_region = np.zeros((H, W), dtype=bool)

print("✓ Legacy broad watermark overwrite disabled")

rgb_v010 = render_taxonomy(
    final_v010
)

original_np = np.array(
    image
)

overlay_v010 = (
    original_np.copy()
)

classified = (
    final_v010
    !=
    UNKNOWN_ID
)


overlay_v010[
    classified
] = (
    original_np[
        classified
    ] * 0.42
    +
    rgb_v010[
        classified
    ] * 0.58
).astype(
    np.uint8
)


plt.figure(
    figsize=(21, 8)
)


plt.subplot(
    1,
    3,
    1
)

plt.imshow(image)

plt.title(
    f"{TEST_ID} — Original"
)

plt.axis("off")


plt.subplot(
    1,
    3,
    2
)

plt.imshow(
    rgb_v010
)

plt.title(
    f"{TEST_ID} — Hybrid v0.10"
)

plt.axis("off")


plt.subplot(
    1,
    3,
    3
)

plt.imshow(
    overlay_v010
)

plt.title(
    f"{TEST_ID} — v0.10 Overlay"
)

plt.axis("off")


plt.tight_layout()
plt.show()


## Stable precision refinements

The following cells are retained because they are part of the latest stable single-image result.  
They refine the base hybrid output without using the later experimental unknown-recovery or batch-replay logic.


In [ ]:
# ============================================================
# STABLE REFINEMENT STATE — v1.3.1 AUDITED
#
# The complete 30-class taxonomy is declared BEFORE inference.
# No class is dynamically appended during refinement.
# This keeps runtime IDs, exported config, and statistics identical.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
import gc

if "final_v010_before_edits" not in globals():
    final_v010_before_edits = final_v010.copy()
    print("✓ Original v0.10-derived result backed up in RAM")
else:
    print("✓ Original v0.10-derived backup already exists")

final_working = final_v010.copy()

TRAFFIC_CONTROL_NAME = "traffic_cone_barrel"
TRAFFIC_CONTROL_ID = LABEL2ID[TRAFFIC_CONTROL_NAME]

# Hard schema assertions: no runtime mutation is allowed here.
assert TRAFFIC_CONTROL_ID == 29
assert CLASS_NAMES[29] == "traffic_cone_barrel"
assert NUM_CLASSES == 30
assert set(CLASS_NAMES) == set(PALETTE)

print("Traffic-control class ID:", TRAFFIC_CONTROL_ID)
print("Declared taxonomy classes:", NUM_CLASSES)
print("✓ Working state ready — taxonomy is frozen before refinement")


In [ ]:
# ============================================================
# TREE LOGIC UPDATE
#
# Goal:
# foliage       -> tree
# branches      -> tree
# main trunk    -> tree
#
# DINO/SAM proposes tree regions,
# ADE + Mapillary provide semantic support.
# ============================================================


TREE_ID = LABEL2ID[
    "tree"
]

TRUNK_ID = LABEL2ID[
    "tree"
]

ROAD_ID = LABEL2ID[
    "roadway"
]

SIDEWALK_ID = LABEL2ID[
    "sidewalk"
]

SKY_ID = LABEL2ID[
    "sky"
]

UPPER_FACADE_ID = LABEL2ID[
    "upper_building_facade"
]

GROUND_SOLID_ID = LABEL2ID[
    "ground_floor_solid_facade"
]

UPPER_GLASS_ID = LABEL2ID[
    "upper_building_glazing"
]

GROUND_GLASS_ID = LABEL2ID[
    "ground_floor_glazing"
]


# ------------------------------------------------------------
# Mapillary vegetation support
# ------------------------------------------------------------

MAP_VEGETATION = (
    mapillary_mask(
        ["vegetation"]
    )
)


TREE_SEMANTIC_SUPPORT = (
    ADE_TREE_V2
    |
    ADE_PLANT_V2
    |
    MAP_VEGETATION
)


# Small tolerance:
# thin branches may be slightly outside
# semantic segmentation boundaries.

tree_support_kernel = (
    cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (
            7,
            7
        )
    )
)


TREE_SUPPORT_DILATED = (
    cv2.dilate(
        TREE_SEMANTIC_SUPPORT
        .astype(
            np.uint8
        ),
        tree_support_kernel,
        iterations=1
    )
    >
    0
)


print(
    "ADE / Mapillary tree support:",
    TREE_SEMANTIC_SUPPORT.mean()
    *
    100,
    "%"
)


# ------------------------------------------------------------
# Reconstruct raw foliage / woody masks
# from existing v0.10 tree_objects
# ------------------------------------------------------------

raw_foliage = np.zeros(
    (
        H,
        W
    ),
    dtype=bool
)

raw_woody = np.zeros(
    (
        H,
        W
    ),
    dtype=bool
)


for obj in tree_objects:

    if (
        obj["component"]
        ==
        "foliage"
        and
        obj["score"]
        >=
        0.27
    ):

        raw_foliage |= (
            obj["mask"]
        )


    elif (
        obj["component"]
        ==
        "woody"
        and
        obj["score"]
        >=
        0.18
    ):

        raw_woody |= (
            obj["mask"]
        )


# ------------------------------------------------------------
# Semantic gate
# ------------------------------------------------------------

gated_foliage = (
    raw_foliage
    &
    TREE_SUPPORT_DILATED
)

gated_woody = (
    raw_woody
    &
    TREE_SUPPORT_DILATED
)


invalid_tree_surface = (
    MAP_ROAD
    |
    MAP_BIKE_LANE
    |
    MAP_SKY
)


gated_foliage &= (
    ~invalid_tree_surface
)

gated_woody &= (
    ~invalid_tree_surface
)


# ------------------------------------------------------------
# Protect important foreground objects
# ------------------------------------------------------------

protected_tree_overlap = np.isin(
    final_v09,
    [
        LABEL2ID[
            "signboard"
        ],
        LABEL2ID[
            "door_entrance"
        ],
        LABEL2ID[
            "sidewalk_shed_scaffold"
        ],
        LABEL2ID[
            "awning_canopy"
        ],
        LABEL2ID[
            "person"
        ],
        LABEL2ID[
            "vehicle"
        ]
    ]
)


gated_foliage &= (
    ~protected_tree_overlap
)

gated_woody &= (
    ~protected_tree_overlap
)


# ------------------------------------------------------------
# Extract MAIN TRUNK from woody mask
#
# The old method:
# all woody -> trunk
#
# New method:
# vertical + thick woody structure -> trunk
# remaining woody structure -> canopy / branch mass
# ------------------------------------------------------------

woody_uint8 = (
    gated_woody
    .astype(
        np.uint8
    )
)


vertical_kernel_height = max(
    15,
    int(
        H * 0.035
    )
)


vertical_kernel = (
    cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (
            3,
            vertical_kernel_height
        )
    )
)


vertical_woody = (
    cv2.morphologyEx(
        woody_uint8,
        cv2.MORPH_OPEN,
        vertical_kernel
    )
    >
    0
)


distance = (
    cv2.distanceTransform(
        woody_uint8,
        cv2.DIST_L2,
        5
    )
)


thickness_threshold = max(
    1.5,
    min(
        H,
        W
    )
    *
    0.0015
)


thick_woody = (
    distance
    >=
    thickness_threshold
)


trunk_seed = (
    vertical_woody
    &
    thick_woody
)


# ------------------------------------------------------------
# Keep only vertically meaningful trunk components
# ------------------------------------------------------------

num_labels, labels_cc, stats_cc, centroids_cc = (
    cv2.connectedComponentsWithStats(
        trunk_seed.astype(
            np.uint8
        ),
        connectivity=8
    )
)


trunk_core = np.zeros(
    (
        H,
        W
    ),
    dtype=bool
)


for component_id in range(
    1,
    num_labels
):

    x = stats_cc[
        component_id,
        cv2.CC_STAT_LEFT
    ]

    y = stats_cc[
        component_id,
        cv2.CC_STAT_TOP
    ]

    w = stats_cc[
        component_id,
        cv2.CC_STAT_WIDTH
    ]

    h = stats_cc[
        component_id,
        cv2.CC_STAT_HEIGHT
    ]

    area = stats_cc[
        component_id,
        cv2.CC_STAT_AREA
    ]


    vertical_ratio = (
        h /
        max(
            w,
            1
        )
    )


    bottom_y = (
        y + h
    )


    # Main trunk should:
    # - be vertically elongated
    # - have meaningful height
    # - extend toward eye/street level

    if (
        h
        >=
        H * 0.045
        and
        vertical_ratio
        >=
        1.8
        and
        bottom_y
        >=
        H * 0.38
    ):

        trunk_core[
            labels_cc
            ==
            component_id
        ] = True


# Restore a little width around trunk core

trunk_restore_kernel = (
    cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (
            9,
            9
        )
    )
)


trunk_refined = (
    cv2.dilate(
        trunk_core.astype(
            np.uint8
        ),
        trunk_restore_kernel,
        iterations=1
    )
    >
    0
)


trunk_refined &= (
    gated_woody
)


# ------------------------------------------------------------
# Remaining woody structure belongs to crown / branch mass
# ------------------------------------------------------------

branch_canopy = (
    gated_woody
    &
    ~trunk_refined
)


canopy_refined = (
    gated_foliage
    |
    branch_canopy
)


# ------------------------------------------------------------
# Rebuild working tree layer from v0.9
# ------------------------------------------------------------

final_working = (
    final_v09.copy()
)


old_tree = np.isin(
    final_working,
    [
        TREE_ID,
        TRUNK_ID
    ]
)


# Remove the old coarse vegetation result first

final_working[
    old_tree
] = UNKNOWN_ID


# Restore obvious underlying surfaces

final_working[
    old_tree
    &
    MAP_ROAD
] = ROAD_ID


final_working[
    old_tree
    &
    MAP_SIDEWALK
] = SIDEWALK_ID


final_working[
    old_tree
    &
    MAP_SKY
] = SKY_ID


# Building background

final_working[
    old_tree
    &
    building_context
] = UPPER_FACADE_ID


# Ground-floor building zone

interface_zone_current = (
    interface_neighborhood_v07
    if
    "interface_neighborhood_v07"
    in globals()
    else
    interface_neighborhood
)


final_working[
    old_tree
    &
    building_context
    &
    interface_zone_current
] = GROUND_SOLID_ID


# Restore glazing when semantic evidence exists

final_working[
    old_tree
    &
    basic_glass
    &
    ~interface_zone_current
] = UPPER_GLASS_ID


final_working[
    old_tree
    &
    basic_glass
    &
    interface_zone_current
] = GROUND_GLASS_ID


# ------------------------------------------------------------
# Install refined vegetation
# ------------------------------------------------------------

final_working[
    canopy_refined
] = TREE_ID


final_working[
    trunk_refined
] = TRUNK_ID


print(
    "Raw woody:",
    raw_woody.mean()
    *
    100,
    "%"
)

print(
    "Semantic-gated woody:",
    gated_woody.mean()
    *
    100,
    "%"
)

print(
    "Refined trunk:",
    trunk_refined.mean()
    *
    100,
    "%"
)

print(
    "Branch mass moved to canopy:",
    branch_canopy.mean()
    *
    100,
    "%"
)

print(
    "✓ Tree logic updated"
)


In [ ]:
# ============================================================
# TRAFFIC CONE / CONSTRUCTION BARREL DETECTION
# ============================================================

from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection,
    Sam2Processor,
    Sam2Model
)

from torchvision.ops import nms


# ------------------------------------------------------------
# Free SAM temporarily before loading DINO
# to avoid T4 VRAM overflow
# ------------------------------------------------------------

if (
    "sam_model"
    in globals()
):

    try:
        del sam_model
    except:
        pass


gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# Load Grounding DINO
# ------------------------------------------------------------

GROUNDING_MODEL_ID = (
    "IDEA-Research/grounding-dino-base"
)


if (
    "grounding_processor"
    not in globals()
):

    grounding_processor = (
        AutoProcessor.from_pretrained(
            GROUNDING_MODEL_ID
        )
    )


grounding_model = (
    AutoModelForZeroShotObjectDetection
    .from_pretrained(
        GROUNDING_MODEL_ID
    )
    .to(device)
)


grounding_model.eval()

print(
    "✓ Grounding DINO loaded for traffic-control detection"
)


TRAFFIC_CONTROL_PROMPTS = [

    "orange traffic cone",
    "road traffic cone",
    "construction traffic cone",
    "construction cone",

    "orange construction barrel",
    "traffic barrel",
    "construction barrel",
    "construction drum"
]


# ------------------------------------------------------------
# Traffic-control ROIs
#
# Focus on street / eye-level zone.
# Overlapping crops help small cones.
# ------------------------------------------------------------

TRAFFIC_ROIS = {

    "left_street": (
        0,
        int(
            H * 0.28
        ),
        int(
            W * 0.58
        ),
        int(
            H * 0.90
        )
    ),

    "center_street": (
        int(
            W * 0.18
        ),
        int(
            H * 0.26
        ),
        int(
            W * 0.84
        ),
        int(
            H * 0.90
        )
    ),

    "right_street": (
        int(
            W * 0.42
        ),
        int(
            H * 0.28
        ),
        W,
        int(
            H * 0.90
        )
    )
}


@torch.no_grad()
def detect_traffic_control(
    crop_image,
    offset_x,
    offset_y
):

    inputs = (
        grounding_processor(
            images=crop_image,
            text=[
                TRAFFIC_CONTROL_PROMPTS
            ],
            return_tensors="pt"
        )
        .to(device)
    )


    outputs = grounding_model(
        **inputs
    )


    results = (
        grounding_processor
        .post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,

            # Cones are small,
            # therefore slightly lower threshold
            threshold=0.18,

            text_threshold=0.16,

            target_sizes=[
                (
                    crop_image.height,
                    crop_image.width
                )
            ]
        )
    )[0]


    detections = []


    for box, score in zip(
        results["boxes"],
        results["scores"]
    ):

        x1, y1, x2, y2 = (
            box.detach()
            .cpu()
            .tolist()
        )


        x1 += offset_x
        x2 += offset_x

        y1 += offset_y
        y2 += offset_y


        box_w = max(
            x2 - x1,
            1
        )

        box_h = max(
            y2 - y1,
            1
        )


        box_area_ratio = (
            box_w
            *
            box_h
            /
            (
                W
                *
                H
            )
        )


        center_y = (
            (
                y1 + y2
            )
            /
            2
            /
            H
        )


        # Reject giant hallucinations

        if (
            box_area_ratio
            >
            0.025
        ):
            continue


        # Reject extremely tiny noise

        if (
            box_area_ratio
            <
            0.00001
        ):
            continue


        # Traffic control should be roughly
        # in street / pedestrian zone

        if not (
            0.28
            <=
            center_y
            <=
            0.92
        ):
            continue


        detections.append({

            "class_name":
                TRAFFIC_CONTROL_NAME,

            "class_id":
                TRAFFIC_CONTROL_ID,

            "score":
                float(
                    score.detach()
                    .cpu()
                    .item()
                ),

            "box": [
                float(x1),
                float(y1),
                float(x2),
                float(y2)
            ]
        })


    return detections


traffic_detections_raw = []


for (
    roi_name,
    roi_box
) in TRAFFIC_ROIS.items():

    x1, y1, x2, y2 = (
        roi_box
    )

    crop = image.crop(
        roi_box
    )

    traffic_detections_raw.extend(

        detect_traffic_control(
            crop,
            x1,
            y1
        )
    )


print(
    "Raw traffic-control detections:",
    len(
        traffic_detections_raw
    )
)


# ------------------------------------------------------------
# NMS
# ------------------------------------------------------------

if (
    len(
        traffic_detections_raw
    )
    >
    0
):

    boxes = torch.tensor(
        [
            d["box"]
            for d
            in traffic_detections_raw
        ],
        dtype=torch.float32
    )


    scores = torch.tensor(
        [
            d["score"]
            for d
            in traffic_detections_raw
        ],
        dtype=torch.float32
    )


    keep = nms(
        boxes,
        scores,
        0.35
    )


    traffic_detections = [

        traffic_detections_raw[
            i
        ]

        for i in keep.tolist()
    ]

else:

    traffic_detections = []


print(
    "After NMS:",
    len(
        traffic_detections
    )
)


# ------------------------------------------------------------
# Release DINO before SAM
# ------------------------------------------------------------

del grounding_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# Load / reload SAM2
# ------------------------------------------------------------

SAM_MODEL_ID = (
    "facebook/sam2.1-hiera-small"
)


sam_processor = (
    Sam2Processor.from_pretrained(
        SAM_MODEL_ID
    )
)


sam_model = (
    Sam2Model.from_pretrained(
        SAM_MODEL_ID
    )
    .to(device)
)


sam_model.eval()

print(
    "✓ SAM2 loaded for traffic-control masks"
)


traffic_objects = (
    boxes_to_masks(
        image,
        traffic_detections
    )
)


print(
    "Traffic-control SAM masks:",
    len(
        traffic_objects
    )
)


In [ ]:
# ============================================================
# TRAFFIC CONTROL QUALITY GATE
# ============================================================


# ------------------------------------------------------------
# Check whether Mapillary itself has a cone-like class
# ------------------------------------------------------------

traffic_mapillary_ids = (
    mapillary_ids(
        [
            "traffic cone",
            "traffic-cone",
            "cone"
        ],
        [
            "traffic sign"
        ]
    )
)


print(
    "Mapillary cone-like classes:"
)


if traffic_mapillary_ids:

    for cid in traffic_mapillary_ids:

        print(
            cid,
            MAP_ID2LABEL[
                cid
            ]
        )

else:

    print(
        "None — DINO/SAM + scene logic will be used."
    )


if traffic_mapillary_ids:

    MAP_TRAFFIC_CONTROL = (
        np.isin(
            map_pred,
            traffic_mapillary_ids
        )
    )

else:

    MAP_TRAFFIC_CONTROL = np.zeros(
        (
            H,
            W
        ),
        dtype=bool
    )


# ------------------------------------------------------------
# Street-surface context
# ------------------------------------------------------------

street_surface = (
    MAP_ROAD
    |
    MAP_SIDEWALK
    |
    MAP_CURB
)


surface_kernel = (
    cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (
            31,
            31
        )
    )
)


street_surface_support = (
    cv2.dilate(
        street_surface.astype(
            np.uint8
        ),
        surface_kernel,
        iterations=1
    )
    >
    0
)


# ------------------------------------------------------------
# Orange color evidence
#
# Used as supporting evidence,
# not as the segmentation itself.
# ------------------------------------------------------------

image_np = np.array(
    image
)


hsv = (
    cv2.cvtColor(
        image_np,
        cv2.COLOR_RGB2HSV
    )
)


h_channel = hsv[
    :,
    :,
    0
]

s_channel = hsv[
    :,
    :,
    1
]

v_channel = hsv[
    :,
    :,
    2
]


orange_support = (
    (
        h_channel
        >=
        3
    )
    &
    (
        h_channel
        <=
        27
    )
    &
    (
        s_channel
        >=
        90
    )
    &
    (
        v_channel
        >=
        65
    )
)


# ------------------------------------------------------------
# Evaluate each SAM proposal
# ------------------------------------------------------------

traffic_control_mask = np.zeros(
    (
        H,
        W
    ),
    dtype=bool
)


accepted_traffic_objects = []


for obj in traffic_objects:

    mask = obj[
        "mask"
    ]


    area = max(
        int(
            mask.sum()
        ),
        1
    )


    area_ratio = (
        area
        /
        (
            H
            *
            W
        )
    )


    ys, xs = np.where(
        mask
    )


    if (
        len(
            ys
        )
        ==
        0
    ):
        continue


    cy = (
        ys.mean()
        /
        H
    )


    # ----------------------------------------
    # Geometry filter
    # ----------------------------------------

    if (
        area_ratio
        >
        0.02
    ):
        continue


    if not (
        0.30
        <=
        cy
        <=
        0.92
    ):
        continue


    # ----------------------------------------
    # Street semantic support
    # ----------------------------------------

    surface_ratio = (
        street_surface_support[
            mask
        ]
        .mean()
    )


    if (
        surface_ratio
        <
        0.25
    ):
        continue


    # ----------------------------------------
    # Orange evidence
    # ----------------------------------------

    orange_ratio = (
        orange_support[
            mask
        ]
        .mean()
    )


    # ----------------------------------------
    # Mapillary evidence, when available
    # ----------------------------------------

    mapillary_ratio = (
        MAP_TRAFFIC_CONTROL[
            mask
        ]
        .mean()
    )


    # Require either orange evidence
    # OR direct Mapillary support.

    if (
        orange_ratio
        <
        0.025
        and
        mapillary_ratio
        <
        0.05
    ):
        continue


    gated_mask = (
        mask
        &
        street_surface_support
        &
        ~MAP_SKY
    )


    traffic_control_mask |= (
        gated_mask
    )


    accepted_traffic_objects.append({

        "score":
            obj["score"],

        "area_ratio":
            area_ratio,

        "surface_ratio":
            float(
                surface_ratio
            ),

        "orange_ratio":
            float(
                orange_ratio
            ),

        "mapillary_ratio":
            float(
                mapillary_ratio
            )
    })


# ------------------------------------------------------------
# Also accept direct Mapillary traffic-control pixels
# if such a class exists
# ------------------------------------------------------------

traffic_control_mask |= (

    MAP_TRAFFIC_CONTROL

    &
    street_surface_support
)


# ------------------------------------------------------------
# Don't overwrite person / vehicle
# ------------------------------------------------------------

protected_dynamic = np.isin(
    final_working,
    [
        LABEL2ID[
            "person"
        ],
        LABEL2ID[
            "vehicle"
        ]
    ]
)


traffic_control_mask &= (
    ~protected_dynamic
)


# ------------------------------------------------------------
# Install class
# ------------------------------------------------------------

final_working[
    traffic_control_mask
] = TRAFFIC_CONTROL_ID


print(
    "Accepted traffic-control objects:",
    len(
        accepted_traffic_objects
    )
)

print(
    "Traffic cone / barrel pixels:",
    int(
        traffic_control_mask.sum()
    )
)

print(
    "Traffic cone / barrel image %:",
    traffic_control_mask.mean()
    *
    100
)


if (
    len(
        accepted_traffic_objects
    )
    >
    0
):

    display(
        pd.DataFrame(
            accepted_traffic_objects
        )
    )


print(
    "✓ Traffic-control class installed"
)


In [ ]:
# ============================================================
# GOOGLE STREET VIEW ARTIFACT DETECTION — v1.2 STRICT IGNORE
#
# SCIENTIFIC RULE:
#   Google Street View screenshot UI / watermark residuals are
#   INVALID EVIDENCE, not scene objects.
#
# Therefore:
# - detect the residual footprint
# - set it to IGNORE=255
# - protect it permanently
# - NEVER assign a taxonomy class back into these pixels
# ============================================================

IGNORE_ID = IGNORE_INDEX if "IGNORE_INDEX" in globals() else 255

image_np = np.array(image)
gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)

# ------------------------------------------------------------
# 1. Search lower portion of Street View screenshot.
#    Dataset artifacts are dark, vertically persistent UI blocks
#    that touch the lower image boundary.
# ------------------------------------------------------------

bottom_search_region = np.zeros((H, W), dtype=bool)
bottom_search_region[int(H * 0.58):, :] = True

very_dark = gray < 58
artifact_dark = gray < 70
candidate_dark = artifact_dark & bottom_search_region

# Close Google lettering / striped internal gaps.
artifact_close_kernel = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (7, 17)
)

candidate_closed = cv2.morphologyEx(
    candidate_dark.astype(np.uint8),
    cv2.MORPH_CLOSE,
    artifact_close_kernel,
    iterations=2
) > 0

# ------------------------------------------------------------
# 2. Connected-component geometry gate
# ------------------------------------------------------------

num_components, artifact_labels, artifact_stats, artifact_centroids = (
    cv2.connectedComponentsWithStats(
        candidate_closed.astype(np.uint8),
        connectivity=8
    )
)

google_artifact_mask = np.zeros((H, W), dtype=bool)
accepted_artifact_components = []

for component_id in range(1, num_components):

    x = int(artifact_stats[component_id, cv2.CC_STAT_LEFT])
    y = int(artifact_stats[component_id, cv2.CC_STAT_TOP])
    w = int(artifact_stats[component_id, cv2.CC_STAT_WIDTH])
    h = int(artifact_stats[component_id, cv2.CC_STAT_HEIGHT])
    area = int(artifact_stats[component_id, cv2.CC_STAT_AREA])

    component_mask = artifact_labels == component_id

    bottom_touch = (y + h) >= (H - 3)
    width_ratio = w / W
    height_ratio = h / H
    area_ratio = area / (H * W)
    fill_ratio = area / max(w * h, 1)
    very_dark_ratio = float(very_dark[component_mask].mean())

    accept = (
        bottom_touch
        and y >= H * 0.55
        and 0.010 <= width_ratio <= 0.18
        and height_ratio >= 0.06
        and area_ratio <= 0.040
        and fill_ratio >= 0.24
        and very_dark_ratio >= 0.35
    )

    if not accept:
        continue

    # --------------------------------------------------------
    # Protect the WHOLE UI footprint, not only dark pixels.
    #
    # v1.2 intentionally uses a slightly more conservative
    # margin than v1.1 to catch antialiased edges and the pale
    # / gray remnants that could otherwise be called "pole".
    # --------------------------------------------------------

    pad_x = max(6, int(round(W * 0.0040)))
    pad_top = max(10, int(round(H * 0.0120)))

    x0 = max(0, x - pad_x)
    x1 = min(W, x + w + pad_x)
    y0 = max(0, y - pad_top)
    y1 = H

    component_bbox_mask = np.zeros((H, W), dtype=bool)
    component_bbox_mask[y0:y1, x0:x1] = True

    google_artifact_mask |= component_bbox_mask

    accepted_artifact_components.append({
        "component": int(component_id),
        "x": x,
        "y": y,
        "width": w,
        "height": h,
        "protected_x0": x0,
        "protected_y0": y0,
        "protected_x1": x1,
        "protected_y1": y1,
        "dark_component_pixels": area,
        "protected_pixels": int(component_bbox_mask.sum()),
        "image_pct": float(component_bbox_mask.mean() * 100.0),
        "very_dark_ratio": very_dark_ratio
    })

# Small final safety expansion for antialiased borders.
artifact_expand_kernel = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (5, 5)
)

google_artifact_mask = cv2.dilate(
    google_artifact_mask.astype(np.uint8),
    artifact_expand_kernel,
    iterations=1
) > 0

artifact_ignore_mask = google_artifact_mask.copy()

# ------------------------------------------------------------
# 3. Install PERMANENT IGNORE
# ------------------------------------------------------------

artifact_pre_ignore_labels = final_working.copy()
final_working[artifact_ignore_mask] = IGNORE_ID

print("Detected Google/UI artifact components:", len(accepted_artifact_components))
print("STRICT IGNORE pixels:", int(artifact_ignore_mask.sum()))
print("STRICT IGNORE image %:", float(artifact_ignore_mask.mean() * 100.0))

if accepted_artifact_components:
    display(pd.DataFrame(accepted_artifact_components))

# ------------------------------------------------------------
# 4. Visual QA
# ------------------------------------------------------------

artifact_preview = image_np.copy()
artifact_preview[artifact_ignore_mask] = (255, 0, 255)

plt.figure(figsize=(18, 8))

plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(artifact_preview)
plt.title("Magenta = PERMANENT Google/UI IGNORE")
plt.axis("off")

plt.tight_layout()
plt.show()

print("✓ Google/UI artifact is permanently IGNORE=255")
print("✓ These pixels must never contribute to pole / road / sidewalk / any taxonomy class")


In [ ]:

# ============================================================
# COMMIT STABLE PRE-REFINEMENT RESULT
#
# Required state boundary before scaffold / tree / pole fixes.
# No intermediate files are exported here.
# ============================================================

final_v010 = final_working.copy()

print("✓ Stable pre-refinement result committed in memory")


In [ ]:
# ============================================================
# SCAFFOLD PRECISION FIX — v1.3 GLASS CURTAIN-WALL VETO
#
# Goal:
# - retain real temporary sidewalk shed / scaffold
# - reject glass-curtain-wall / glazed-façade false positives
# - restore rejected glass-like pixels to glazing, not UNKNOWN
# ============================================================

final_working = final_v010.copy()

SCAFFOLD_ID = LABEL2ID["sidewalk_shed_scaffold"]
ROAD_ID = LABEL2ID["roadway"]
SIDEWALK_ID = LABEL2ID["sidewalk"]
CURB_ID = LABEL2ID["curb_edge"]
UPPER_FACADE_ID = LABEL2ID["upper_building_facade"]
UPPER_GLASS_ID = LABEL2ID["upper_building_glazing"]
GROUND_GLASS_ID = LABEL2ID["ground_floor_glazing"]

current_scaffold = final_working == SCAFFOLD_ID
print("Scaffold before:", int(current_scaffold.sum()), "pixels")

building_support = MAP_BUILDING | ADE_BUILDING_V2
building_near = cv2.dilate(
    building_support.astype(np.uint8),
    cv2.getStructuringElement(cv2.MORPH_RECT, (61, 41)),
    iterations=1
) > 0

# Strong glass evidence from three independent paths.
glazing_support = (
    (ADE_GLASS_V2 | MAP_WINDOW | precise_window_mask | facade_glass_aggressive)
    & building_support
)

# Stronger DINO scaffold evidence than the original fusion threshold.
strong_scaffold_dino = np.zeros((H, W), dtype=bool)
for obj in detail_objects:
    if (
        obj["class_name"] == "sidewalk_shed_scaffold"
        and obj["score"] >= 0.32
    ):
        strong_scaffold_dino |= obj["mask"]

num_labels, labels_scaf, stats_scaf, _ = cv2.connectedComponentsWithStats(
    current_scaffold.astype(np.uint8), connectivity=8
)

accepted_scaffold = np.zeros((H, W), dtype=bool)
scaffold_diagnostics = []

for cid in range(1, num_labels):
    component = labels_scaf == cid
    area = int(component.sum())
    if area == 0:
        continue

    x = int(stats_scaf[cid, cv2.CC_STAT_LEFT])
    y = int(stats_scaf[cid, cv2.CC_STAT_TOP])
    w = int(stats_scaf[cid, cv2.CC_STAT_WIDTH])
    h = int(stats_scaf[cid, cv2.CC_STAT_HEIGHT])

    area_ratio = area / (H * W)
    building_ratio = float(building_near[component].mean())
    sidewalk_ratio = float(MAP_SIDEWALK[component].mean())
    glazing_ratio = float(glazing_support[component].mean())
    strong_dino_ratio = float(strong_scaffold_dino[component].mean())
    cy = (y + h / 2) / H
    vertical_ratio = h / max(w, 1)

    # Typical curtain-wall false positive:
    # vertically tall + glass-dominant + little sidewalk evidence.
    glass_tower_like = (
        glazing_ratio >= 0.22
        and vertical_ratio >= 1.10
        and sidewalk_ratio < 0.08
    )

    glass_panel_like = (
        glazing_ratio >= 0.42
        and strong_dino_ratio < 0.18
    )

    scaffold_evidence = (
        strong_dino_ratio >= 0.10
        or sidewalk_ratio >= 0.04
    )

    accept = (
        building_ratio >= 0.22
        and area_ratio <= 0.08
        and 0.20 <= cy <= 0.88
        and scaffold_evidence
        and not glass_tower_like
        and not glass_panel_like
    )

    if accept:
        accepted_scaffold[component] = True

    scaffold_diagnostics.append({
        "component": cid,
        "pixels": area,
        "image_pct": area_ratio * 100,
        "building_support": building_ratio,
        "sidewalk_support": sidewalk_ratio,
        "glazing_support": glazing_ratio,
        "strong_scaffold_dino": strong_dino_ratio,
        "vertical_ratio": vertical_ratio,
        "center_y": cy,
        "glass_tower_veto": glass_tower_like,
        "glass_panel_veto": glass_panel_like,
        "accepted": accept,
    })

rejected_scaffold = current_scaffold & ~accepted_scaffold
print("Accepted scaffold:", int(accepted_scaffold.sum()))
print("Rejected scaffold:", int(rejected_scaffold.sum()))
if scaffold_diagnostics:
    display(pd.DataFrame(scaffold_diagnostics))

# Reset rejected scaffold first.
final_working[rejected_scaffold] = UNKNOWN_ID

# Restore reliable street surfaces.
final_working[rejected_scaffold & MAP_ROAD] = ROAD_ID
final_working[rejected_scaffold & MAP_SIDEWALK] = SIDEWALK_ID
final_working[rejected_scaffold & MAP_CURB] = CURB_ID

# Restore building surface. Glazing gets priority over opaque façade.
rejected_building = rejected_scaffold & building_support
final_working[rejected_building] = UPPER_FACADE_ID

rejected_glass = rejected_scaffold & glazing_support

ground_glass_recovery = rejected_glass & interface_neighborhood
upper_glass_recovery = rejected_glass & ~interface_neighborhood

final_working[ground_glass_recovery] = GROUND_GLASS_ID
final_working[upper_glass_recovery] = UPPER_GLASS_ID

# Keep only scaffold components that survive the glass veto.
final_working[accepted_scaffold] = SCAFFOLD_ID

print("Recovered rejected scaffold as upper glazing:", int(upper_glass_recovery.sum()))
print("Recovered rejected scaffold as ground glazing:", int(ground_glass_recovery.sum()))
print("✓ v1.3 scaffold glass-curtain-wall safeguard applied")


In [ ]:
# ============================================================
# FENCE / RAILING RECOVERY — v1.3
#
# Purpose:
# Recover park perimeter fences / railings that remain
# other_unknown, without performing generic unknown filling.
# ============================================================

FENCE_ID = LABEL2ID["fence_railing"]

# 1. Semantic-model evidence
fence_semantic = MAP_FENCE | ADE_FENCE_V3

# 2. DINO + SAM fence evidence
fence_dino = np.zeros((H, W), dtype=bool)
for obj in detail_objects:
    if (
        obj["class_name"] == "fence_railing"
        and obj["score"] >= 0.18
    ):
        fence_dino |= obj["mask"]

# 3. Park/street-edge context: fence often runs next to sidewalk.
sidewalk_near_fence = cv2.dilate(
    MAP_SIDEWALK.astype(np.uint8),
    cv2.getStructuringElement(cv2.MORPH_RECT, (101, 61)),
    iterations=1
) > 0

fence_candidate = (
    fence_semantic
    |
    (fence_dino & sidewalk_near_fence)
)

# Fence cannot be road, sky, glass façade, or Google UI artifact.
fence_candidate &= ~MAP_ROAD
fence_candidate &= ~MAP_SKY
fence_candidate &= ~ADE_GLASS_V2
fence_candidate &= ~MAP_WINDOW

if "artifact_ignore_mask" in globals():
    fence_candidate &= ~artifact_ignore_mask

# Preserve already reliable foreground/object classes.
protected_fence_classes = np.isin(
    final_working,
    [
        LABEL2ID["person"],
        LABEL2ID["vehicle"],
        LABEL2ID["traffic_cone_barrel"],
        LABEL2ID["traffic_sign_signal"],
        LABEL2ID["sidewalk_shed_scaffold"],
        IGNORE_ID,
    ]
)
fence_candidate &= ~protected_fence_classes

# Deliberately conservative: v1.3 recovery only converts pixels
# that are still unknown (plus any already-fence pixels).
fence_recovery = (
    fence_candidate
    & np.isin(final_working, [UNKNOWN_ID, FENCE_ID])
)

# Remove tiny isolated speckle while retaining thin openwork fence lines.
num_f, lab_f, stats_f, _ = cv2.connectedComponentsWithStats(
    fence_recovery.astype(np.uint8), connectivity=8
)
fence_precise = np.zeros((H, W), dtype=bool)
fence_rows = []

for cid in range(1, num_f):
    comp = lab_f == cid
    area = int(comp.sum())
    x = int(stats_f[cid, cv2.CC_STAT_LEFT])
    y = int(stats_f[cid, cv2.CC_STAT_TOP])
    w = int(stats_f[cid, cv2.CC_STAT_WIDTH])
    h = int(stats_f[cid, cv2.CC_STAT_HEIGHT])
    cy = (y + h/2) / H
    area_ratio = area / (H*W)
    semantic_ratio = float(fence_semantic[comp].mean()) if area else 0.0
    dino_ratio = float(fence_dino[comp].mean()) if area else 0.0

    accept = (
        area >= 8
        and area_ratio <= 0.035
        and 0.18 <= cy <= 0.90
        and (semantic_ratio >= 0.05 or dino_ratio >= 0.05)
    )

    if accept:
        fence_precise[comp] = True

    fence_rows.append({
        "component": cid,
        "pixels": area,
        "width": w,
        "height": h,
        "center_y": cy,
        "semantic_support": semantic_ratio,
        "dino_support": dino_ratio,
        "accepted": accept,
    })

unknown_before_fence = int(np.sum(final_working == UNKNOWN_ID))
final_working[fence_precise] = FENCE_ID
unknown_after_fence = int(np.sum(final_working == UNKNOWN_ID))

print("Fence semantic pixels:", int(fence_semantic.sum()))
print("Fence DINO/SAM pixels:", int(fence_dino.sum()))
print("Recovered fence pixels:", int(fence_precise.sum()))
print("Unknown reduction from fence recovery:", unknown_before_fence - unknown_after_fence)
if fence_rows:
    display(pd.DataFrame(fence_rows))
print("✓ v1.3 fence / railing recovery applied")


In [ ]:
# ============================================================
# TREE PRECISION UPDATE
#
# Output taxonomy remains:
#
# tree =
#   foliage + visible branch structure
#
# tree =
#   main visible trunk
#
# IMPORTANT:
# We do NOT create a separate branch taxonomy class.
# ============================================================

TREE_ID = LABEL2ID[
    "tree"
]

TRUNK_ID = LABEL2ID[
    "tree"
]


# ------------------------------------------------------------
# Save foreground classes that tree must never overwrite
# ------------------------------------------------------------

PROTECTED_IDS = [

    LABEL2ID["person"],
    LABEL2ID["vehicle"],
    LABEL2ID["signboard"],
    LABEL2ID["door_entrance"],
    LABEL2ID["awning_canopy"],
    LABEL2ID["sidewalk_shed_scaffold"],
    LABEL2ID["traffic_cone_barrel"],
    IGNORE_ID
]


protected_foreground = np.isin(
    final_working,
    PROTECTED_IDS
)

# Street View screenshot artifacts are protected from tree recovery.
if "google_artifact_mask" in globals():
    protected_foreground |= google_artifact_mask


# ------------------------------------------------------------
# Reconstruct raw foliage / woody proposals
# from the already-computed tree_objects
# ------------------------------------------------------------

raw_foliage = np.zeros(
    (H, W),
    dtype=bool
)

raw_woody = np.zeros(
    (H, W),
    dtype=bool
)


for obj in tree_objects:

    if (
        obj["component"]
        ==
        "foliage"
        and
        obj["score"]
        >=
        0.27
    ):

        raw_foliage |= obj[
            "mask"
        ]


    elif (
        obj["component"]
        ==
        "woody"
        and
        obj["score"]
        >=
        0.18
    ):

        raw_woody |= obj[
            "mask"
        ]


# ------------------------------------------------------------
# Exact semantic tree support
#
# NOTE:
# Previous patch used 7x7 dilation.
# That made the leafless crown too blob-like.
#
# This time we DO NOT expand vegetation support first.
# ------------------------------------------------------------

MAP_VEGETATION = (
    mapillary_mask(
        ["vegetation"]
    )
)


TREE_SEMANTIC_CORE = (

    ADE_TREE_V2

    |

    ADE_PLANT_V2

    |

    MAP_VEGETATION
)


# ------------------------------------------------------------
# Invalid tree surfaces
# ------------------------------------------------------------

invalid_tree_surface = (

    MAP_ROAD

    |

    MAP_BIKE_LANE

    |

    MAP_SKY
)


# ------------------------------------------------------------
# FOLIAGE
#
# Foliage can be area-like, so semantic vegetation support
# is appropriate.
# ------------------------------------------------------------

foliage_precise = (

    raw_foliage

    &

    TREE_SEMANTIC_CORE

    &

    ~invalid_tree_surface

    &

    ~protected_foreground
)


# ------------------------------------------------------------
# WOODY / BRANCH candidate
# ------------------------------------------------------------

woody_candidate = (

    raw_woody

    &

    TREE_SEMANTIC_CORE

    &

    ~invalid_tree_surface

    &

    ~protected_foreground
)


# ============================================================
# Branch-structure extraction from ORIGINAL IMAGE
#
# Leafless branches are line-like / locally dark structures.
#
# SAM box interior should NOT automatically become tree.
# ============================================================

image_np = np.array(
    image
)


gray = cv2.cvtColor(
    image_np,
    cv2.COLOR_RGB2GRAY
)


# ------------------------------------------------------------
# 1. Image edges
# ------------------------------------------------------------

edges = cv2.Canny(
    gray,
    45,
    125
)


edge_kernel = (
    cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5, 5)
    )
)


edge_support = (
    cv2.dilate(
        edges,
        edge_kernel,
        iterations=1
    )
    >
    0
)


# ------------------------------------------------------------
# 2. Dark thin-structure enhancement
#
# Black-hat highlights dark branches against brighter sky /
# building background.
# ------------------------------------------------------------

blackhat_kernel = (
    cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (11, 11)
    )
)


blackhat = cv2.morphologyEx(
    gray,
    cv2.MORPH_BLACKHAT,
    blackhat_kernel
)


dark_line_support = (
    blackhat
    >
    10
)


# ------------------------------------------------------------
# Branch pixels need visible image structure.
# This prevents a SAM-filled crown blob.
# ------------------------------------------------------------

branch_structure_support = (

    edge_support

    |

    dark_line_support
)


branch_precise = (

    woody_candidate

    &

    branch_structure_support
)


# ------------------------------------------------------------
# Clean isolated one-pixel noise
# ------------------------------------------------------------

branch_clean_kernel = (
    cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (3, 3)
    )
)


branch_precise = (
    cv2.morphologyEx(
        branch_precise.astype(np.uint8),
        cv2.MORPH_CLOSE,
        branch_clean_kernel,
        iterations=1
    )
    >
    0
)


# ============================================================
# MAIN TRUNK
#
# Extract thicker + vertically persistent regions from woody
# proposal, independently from branch classification.
# ============================================================

woody_uint8 = (
    woody_candidate.astype(
        np.uint8
    )
)


distance = (
    cv2.distanceTransform(
        woody_uint8,
        cv2.DIST_L2,
        5
    )
)


thick_woody = (
    distance
    >=
    2.0
)


vertical_kernel_h = max(
    17,
    int(
        H * 0.045
    )
)


vertical_kernel = (
    cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (
            3,
            vertical_kernel_h
        )
    )
)


vertical_woody = (
    cv2.morphologyEx(
        woody_uint8,
        cv2.MORPH_OPEN,
        vertical_kernel
    )
    >
    0
)


trunk_seed = (
    thick_woody
    &
    vertical_woody
)


num_labels, labels_tree, stats_tree, _ = (
    cv2.connectedComponentsWithStats(
        trunk_seed.astype(np.uint8),
        connectivity=8
    )
)


trunk_core = np.zeros(
    (H, W),
    dtype=bool
)


for cid in range(
    1,
    num_labels
):

    x = stats_tree[
        cid,
        cv2.CC_STAT_LEFT
    ]

    y = stats_tree[
        cid,
        cv2.CC_STAT_TOP
    ]

    w = stats_tree[
        cid,
        cv2.CC_STAT_WIDTH
    ]

    h = stats_tree[
        cid,
        cv2.CC_STAT_HEIGHT
    ]


    vertical_ratio = (
        h
        /
        max(
            w,
            1
        )
    )


    bottom = (
        y + h
    )


    if (

        h
        >=
        H * 0.055

        and

        vertical_ratio
        >=
        1.8

        and

        bottom
        >=
        H * 0.38
    ):

        trunk_core[
            labels_tree
            ==
            cid
        ] = True


# Restore some actual trunk width

trunk_expand_kernel = (
    cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (7, 7)
    )
)


trunk_precise = (
    cv2.dilate(
        trunk_core.astype(np.uint8),
        trunk_expand_kernel,
        iterations=1
    )
    >
    0
)


trunk_precise &= (
    woody_candidate
)


trunk_precise &= (
    ~protected_foreground
)


# Trunk wins over branch at overlap

branch_precise &= (
    ~trunk_precise
)


# ------------------------------------------------------------
# FINAL taxonomy interpretation:
#
# foliage + visible branch pixels = tree
# main stem = tree
# ------------------------------------------------------------

tree_precise = (

    foliage_precise

    |

    branch_precise
)


# ============================================================
# Remove CURRENT bad tree layer
# ============================================================

old_tree_pixels = np.isin(
    final_working,
    [
        TREE_ID,
        TRUNK_ID
    ]
)


final_working[
    old_tree_pixels
] = UNKNOWN_ID


# ------------------------------------------------------------
# Restore underlying semantic surfaces
# ------------------------------------------------------------

final_working[
    old_tree_pixels
    &
    MAP_ROAD
] = ROAD_ID


final_working[
    old_tree_pixels
    &
    MAP_SIDEWALK
] = SIDEWALK_ID


final_working[
    old_tree_pixels
    &
    MAP_CURB
] = CURB_ID


final_working[
    old_tree_pixels
    &
    MAP_SKY
] = LABEL2ID[
    "sky"
]


final_working[
    old_tree_pixels
    &
    building_context
] = UPPER_FACADE_ID


# Preserve known glazing underneath tree mistake

final_working[
    old_tree_pixels
    &
    basic_glass
    &
    ~interface_zone_current
] = LABEL2ID[
    "upper_building_glazing"
]


final_working[
    old_tree_pixels
    &
    basic_glass
    &
    interface_zone_current
] = LABEL2ID[
    "ground_floor_glazing"
]


# ------------------------------------------------------------
# Install corrected tree masks
# ------------------------------------------------------------

final_working[
    tree_precise
] = TREE_ID


final_working[
    trunk_precise
] = TRUNK_ID


# ------------------------------------------------------------
# Restore important foreground
# from result before tree rewrite
# ------------------------------------------------------------

previous_result = (
    final_v010.copy()
)


final_working[
    protected_foreground
] = previous_result[
    protected_foreground
]


print()
print(
    "Raw foliage %:",
    raw_foliage.mean()
    *
    100
)

print(
    "Precise foliage %:",
    foliage_precise.mean()
    *
    100
)

print()

print(
    "Raw woody %:",
    raw_woody.mean()
    *
    100
)

print(
    "Branch structure %:",
    branch_precise.mean()
    *
    100
)

print(
    "Main trunk %:",
    trunk_precise.mean()
    *
    100
)

print()

print(
    "Final tree %:",
    tree_precise.mean()
    *
    100
)

print(
    "✓ Tree geometry rebuilt"
)


In [ ]:
# ============================================================
# POLE + TRAFFIC SIGNAL RECOVERY
#
# pole_fixture:
# - streetlight pole
# - utility pole
# - traffic-signal support pole / mast
#
# traffic_sign_signal:
# - traffic light head
# - traffic sign
# ============================================================

POLE_ID = LABEL2ID[
    "pole_fixture"
]

TRAFFIC_SIGNAL_ID = LABEL2ID[
    "traffic_sign_signal"
]


# ============================================================
# MAPILLARY SUPPORT
# ============================================================

MAP_POLE = mapillary_mask(
    [
        "pole"
    ]
)


MAP_TRAFFIC_LIGHT = mapillary_mask(
    [
        "traffic light"
    ]
)


MAP_TRAFFIC_SIGN = mapillary_mask(
    [
        "traffic sign"
    ]
)


print(
    "Mapillary pole:",
    int(
        MAP_POLE.sum()
    )
)

print(
    "Mapillary traffic light:",
    int(
        MAP_TRAFFIC_LIGHT.sum()
    )
)

print(
    "Mapillary traffic sign:",
    int(
        MAP_TRAFFIC_SIGN.sum()
    )
)


# ============================================================
# ADE SUPPORT
# ============================================================

ADE_POLE_CURRENT = ade_mask_fuzzy(
    [
        "pole",
        "streetlight",
        "street light"
    ]
)


ADE_TRAFFIC_LIGHT_CURRENT = ade_mask_fuzzy(
    [
        "traffic light"
    ]
)


ADE_TRAFFIC_SIGN_CURRENT = ade_mask_fuzzy(
    [
        "traffic sign"
    ]
)


# ------------------------------------------------------------
# Combined proposal
# ------------------------------------------------------------

pole_candidate = (

    MAP_POLE

    |

    ADE_POLE_CURRENT
)


traffic_signal_candidate = (

    MAP_TRAFFIC_LIGHT

    |

    MAP_TRAFFIC_SIGN

    |

    ADE_TRAFFIC_LIGHT_CURRENT

    |

    ADE_TRAFFIC_SIGN_CURRENT
)


# ============================================================
# POLE GEOMETRY FILTER
#
# Pole should generally be a vertically persistent component.
# ============================================================

num_poles, pole_labels, pole_stats, _ = (
    cv2.connectedComponentsWithStats(
        pole_candidate.astype(np.uint8),
        connectivity=8
    )
)


pole_precise = np.zeros(
    (H, W),
    dtype=bool
)


pole_rows = []


for cid in range(
    1,
    num_poles
):

    component = (
        pole_labels
        ==
        cid
    )


    area = int(
        component.sum()
    )


    x = pole_stats[
        cid,
        cv2.CC_STAT_LEFT
    ]

    y = pole_stats[
        cid,
        cv2.CC_STAT_TOP
    ]

    w = pole_stats[
        cid,
        cv2.CC_STAT_WIDTH
    ]

    h = pole_stats[
        cid,
        cv2.CC_STAT_HEIGHT
    ]


    area_ratio = (
        area
        /
        (H * W)
    )


    vertical_ratio = (
        h
        /
        max(
            w,
            1
        )
    )


    # Allow:
    # - long vertical poles
    # - smaller distant poles

    accept = (

        area_ratio
        <=
        0.018

        and

        h
        >=
        H * 0.025

        and

        vertical_ratio
        >=
        1.35
    )


    if accept:

        pole_precise[
            component
        ] = True


    pole_rows.append({

        "component":
            cid,

        "pixels":
            area,

        "width":
            w,

        "height":
            h,

        "vertical_ratio":
            vertical_ratio,

        "accepted":
            accept
    })


if pole_rows:

    display(
        pd.DataFrame(
            pole_rows
        )
    )


# ------------------------------------------------------------
# Never call road / sidewalk pixels pole unless semantic model
# explicitly returned the pole there.
# ------------------------------------------------------------

pole_precise &= (
    ~MAP_SKY
)


# ------------------------------------------------------------
# Protect dynamic / research objects
# ------------------------------------------------------------

do_not_overwrite = np.isin(
    final_working,
    [

        LABEL2ID[
            "person"
        ],

        LABEL2ID[
            "vehicle"
        ],

        LABEL2ID[
            "traffic_cone_barrel"
        ],

        IGNORE_ID
    ]
)


pole_precise &= (
    ~do_not_overwrite
)


traffic_signal_candidate &= (
    ~do_not_overwrite
)

# Absolute protection: a detected Street View screenshot artifact
# must never be reclassified as a pole / sign / signal.
if "google_artifact_mask" in globals():
    pole_precise &= ~google_artifact_mask
    traffic_signal_candidate &= ~google_artifact_mask


# ------------------------------------------------------------
# Install pole first
# ------------------------------------------------------------

final_working[
    pole_precise
] = POLE_ID


# Signal/sign head has priority over its pole

final_working[
    traffic_signal_candidate
] = TRAFFIC_SIGNAL_ID


print(
    "Final pole pixels:",
    int(
        pole_precise.sum()
    )
)

print(
    "Final traffic sign/signal pixels:",
    int(
        traffic_signal_candidate.sum()
    )
)

print(
    "✓ Pole / traffic signal recovery applied"
)


In [ ]:
# ============================================================
# STREET VIEW ARTIFACT FINAL LOCK — v1.2 STRICT IGNORE
#
# IMPORTANT:
# Scientific data:
#   artifact pixels stay IGNORE=255 permanently.
#
# Display only:
#   create a separate contextual visualization fill so the
#   ignored screenshot block does not appear as a gray/black
#   object in the figure.
#
# The display fill is NEVER copied back into final_working.
# ============================================================

from scipy.ndimage import distance_transform_edt

if "google_artifact_mask" not in globals():
    google_artifact_mask = np.zeros((H, W), dtype=bool)

artifact_ignore_mask = google_artifact_mask.copy()
artifact_detected_pixels = int(artifact_ignore_mask.sum())

# ------------------------------------------------------------
# 1. HARD FINAL LOCK
#
# Even if an earlier/later refinement accidentally touched the
# area, this line wins immediately before final commit.
# ------------------------------------------------------------

final_working[artifact_ignore_mask] = IGNORE_ID

# Compatibility audit variables.
artifact_restored_mask = np.zeros((H, W), dtype=bool)
artifact_unresolved_mask = artifact_ignore_mask.copy()

# ------------------------------------------------------------
# 2. DISPLAY-ONLY contextual fill
#
# Use only broad scene-surface / vegetation classes as donors.
# Explicitly EXCLUDE narrow object classes that could recreate
# the screenshot shape:
#   pole, sign/signal, person, vehicle, cone, signboard, etc.
# ------------------------------------------------------------

DISPLAY_CONTEXT_CLASS_NAMES = [
    "roadway",
    "sidewalk",
    "bike_lane",
    "curb_edge",
    "upper_building_facade",
    "ground_floor_solid_facade",
    "ground_floor_glazing",
    "upper_building_glazing",
    "door_entrance",
    "awning_canopy",
    "sidewalk_shed_scaffold",
    "tree",
    "shrub_hedge",
    "ground_vegetation",
    "vertical_green_wall",
    "sky",
    "other_unknown",
]

display_context_ids = [
    LABEL2ID[name]
    for name in DISPLAY_CONTEXT_CLASS_NAMES
    if name in LABEL2ID
]

display_context_labels = final_working.copy()

display_donor_mask = (
    ~artifact_ignore_mask
    &
    np.isin(final_working, display_context_ids)
    &
    (final_working != IGNORE_ID)
)

if artifact_detected_pixels > 0 and display_donor_mask.any():

    # nearest ZERO = nearest valid donor
    search_space = ~display_donor_mask

    _, nearest_indices = distance_transform_edt(
        search_space,
        sampling=(2.5, 1.0),  # favor left/right continuity
        return_distances=True,
        return_indices=True
    )

    nearest_y = nearest_indices[0]
    nearest_x = nearest_indices[1]

    contextual_fill = final_working[
        nearest_y,
        nearest_x
    ]

    display_context_labels[artifact_ignore_mask] = contextual_fill[
        artifact_ignore_mask
    ]

else:
    display_context_labels[artifact_ignore_mask] = UNKNOWN_ID

# Absolute display safety:
# even if future edits alter donor logic, Google/UI pixels may
# never display as pole_fixture or traffic_sign_signal.
for forbidden_name in [
    "pole_fixture",
    "traffic_sign_signal",
    "person",
    "vehicle",
    "traffic_cone_barrel",
    "signboard",
]:
    if forbidden_name in LABEL2ID:
        bad_display = (
            artifact_ignore_mask
            &
            (display_context_labels == LABEL2ID[forbidden_name])
        )
        display_context_labels[bad_display] = UNKNOWN_ID

# ------------------------------------------------------------
# 3. HARD QA ASSERTIONS
# ------------------------------------------------------------

assert np.all(
    final_working[artifact_ignore_mask] == IGNORE_ID
), "STRICT IGNORE FAILURE: artifact pixels received semantic labels."

if "pole_fixture" in LABEL2ID:
    assert not np.any(
        artifact_ignore_mask
        &
        (final_working == LABEL2ID["pole_fixture"])
    ), "STRICT IGNORE FAILURE: Google artifact became pole_fixture."

if "traffic_sign_signal" in LABEL2ID:
    assert not np.any(
        artifact_ignore_mask
        &
        (final_working == LABEL2ID["traffic_sign_signal"])
    ), "STRICT IGNORE FAILURE: Google artifact became traffic_sign_signal."

print("========================================")
print("STREET VIEW ARTIFACT FINAL LOCK")
print("========================================")
print("Detected artifact pixels:", artifact_detected_pixels)
print("Scientific semantic assignments inside artifact:", 0)
print("Permanent IGNORE pixels:", int(artifact_ignore_mask.sum()))
print("✓ Artifact cannot contribute to pole_fixture")
print("✓ Artifact cannot contribute to traffic_sign_signal")
print("✓ Artifact is excluded from all class percentages")
print("✓ Contextual fill exists ONLY for visualization")


In [ ]:
# ============================================================
# COMMIT + STRICT IGNORE VISUAL CHECK — v1.2
# ============================================================

final_v010 = final_working.copy()

# Re-assert permanent scientific IGNORE at the commit boundary.
if "artifact_ignore_mask" in globals():
    final_v010[artifact_ignore_mask] = IGNORE_ID

rgb_v010_scientific = render_taxonomy(final_v010)

# Scientific IGNORE shown dark gray in this QA panel only.
rgb_v010_scientific[
    final_v010 == IGNORE_ID
] = (35, 35, 35)

original_np = np.array(image)

# Display-clean semantic map is separate from scientific labels.
display_label_map = (
    display_context_labels.copy()
    if "display_context_labels" in globals()
    else final_v010.copy()
)

display_label_map[
    display_label_map == IGNORE_ID
] = UNKNOWN_ID

rgb_v010 = render_taxonomy(display_label_map)

overlay_v010 = original_np.copy()

normal_classified = (
    (final_v010 != UNKNOWN_ID)
    &
    (final_v010 != IGNORE_ID)
)

overlay_v010[normal_classified] = (
    original_np[normal_classified] * 0.42
    +
    rgb_v010[normal_classified] * 0.58
).astype(np.uint8)

# Artifact area: use display-only contextual semantic color,
# not the original screenshot pixels.
if "artifact_ignore_mask" in globals():
    overlay_v010[artifact_ignore_mask] = rgb_v010[artifact_ignore_mask]

# Hard post-commit assertion.
if "artifact_ignore_mask" in globals():
    assert np.all(final_v010[artifact_ignore_mask] == IGNORE_ID)

plt.figure(figsize=(22, 8))

plt.subplot(1, 3, 1)
plt.imshow(image)
plt.title(f"{TEST_ID} — Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(rgb_v010_scientific)
plt.title(f"{TEST_ID} — Scientific Labels (dark gray = IGNORE)")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(rgb_v010)
plt.title(f"{TEST_ID} — Display Clean (context fill only)")
plt.axis("off")

plt.tight_layout()
plt.show()

print("✓ Scientific map keeps Google/UI residual as IGNORE=255")
print("✓ Display map fills the hole without changing scientific statistics")


In [ ]:
# ============================================================
# CURRENT QA — v1.3.1 AUDITED
# ============================================================

valid = final_v010 != IGNORE_ID
valid_count = max(int(valid.sum()), 1)

def current_pct(class_name):
    cid = LABEL2ID[class_name]
    return np.sum(valid & (final_v010 == cid)) / valid_count * 100

qa_classes = [
    "roadway",
    "sidewalk",
    "sidewalk_shed_scaffold",
    "fence_railing",
    "traffic_cone_barrel",
    "pole_fixture",
    "traffic_sign_signal",
    "tree",
    "upper_building_facade",
    "upper_building_glazing",
    "ground_floor_glazing",
    "other_unknown",
]

qa_df = pd.DataFrame([
    {"class_name": name, "percentage": current_pct(name)}
    for name in qa_classes
])
display(qa_df)

# -----------------------------
# Scientific / schema invariants
# -----------------------------
declared_ids = set(CLASS_NAMES.keys())
observed_ids = set(np.unique(final_v010[valid]).astype(int).tolist())

assert observed_ids.issubset(declared_ids), (
    f"Undeclared label IDs found: {sorted(observed_ids - declared_ids)}"
)
assert NUM_CLASSES == TAXONOMY["num_classes"] == len(CVAT_LABELS) == 30
assert LABEL2ID["traffic_cone_barrel"] == 29
assert "tree_canopy" not in LABEL2ID and "tree_trunk" not in LABEL2ID

if "artifact_ignore_mask" in globals():
    assert np.all(final_v010[artifact_ignore_mask] == IGNORE_ID)

total_pct = sum(
    np.sum(valid & (final_v010 == cid)) / valid_count * 100
    for cid in CLASS_NAMES
)

print("Tree (leaves + branches + trunk):", current_pct("tree"), "%")
print("Fence / railing:", current_pct("fence_railing"), "%")
print("Scaffold:", current_pct("sidewalk_shed_scaffold"), "%")
print("Upper glazing:", current_pct("upper_building_glazing"), "%")
print("Ground glazing:", current_pct("ground_floor_glazing"), "%")
print("Other unknown:", current_pct("other_unknown"), "%")
print("Declared-class percentage sum:", total_pct, "%")

assert abs(total_pct - 100.0) < 1e-6
print("✓ SHARE QA PASSED: schema, labels, IGNORE policy, and percentages are internally consistent")


In [ ]:
# ============================================================
# STRICT-IGNORE DISPLAY — v1.2
#
# SCIENTIFIC:
#   final_v010 keeps Google/UI artifact as IGNORE=255.
#
# DISPLAY:
#   display_label_map uses contextual broad-surface fill only.
#   It is never used for statistics.
# ============================================================

display_label_map = (
    display_context_labels.copy()
    if "display_context_labels" in globals()
    else final_v010.copy()
)

display_label_map[
    display_label_map == IGNORE_ID
] = UNKNOWN_ID

rgb_display_clean = render_taxonomy(display_label_map)

original_np = np.array(image)
overlay_display_clean = original_np.copy()

normal_classified = (
    (final_v010 != UNKNOWN_ID)
    &
    (final_v010 != IGNORE_ID)
)

overlay_display_clean[normal_classified] = (
    original_np[normal_classified] * 0.42
    +
    rgb_display_clean[normal_classified] * 0.58
).astype(np.uint8)

if "artifact_ignore_mask" in globals():
    overlay_display_clean[artifact_ignore_mask] = (
        rgb_display_clean[artifact_ignore_mask]
    )

plt.figure(figsize=(22, 8))

plt.subplot(1, 3, 1)
plt.imshow(image)
plt.title(f"{TEST_ID} — Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(rgb_display_clean)
plt.title(f"{TEST_ID} — Combined Semantic Mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(overlay_display_clean)
plt.title(f"{TEST_ID} — Strict-Ignore Overlay")
plt.axis("off")

plt.tight_layout()
plt.show()

print("✓ Google/UI pixels are excluded from scientific labels")
print("✓ Display fill cannot create pole/sign/person/vehicle/cone artifacts")


In [ ]:
# ============================================================
# USE STRICT-IGNORE DISPLAY VERSION FOR VISUAL EXPORT
# ============================================================

rgb_v010 = rgb_display_clean.copy()
overlay_v010 = overlay_display_clean.copy()

print("✓ RGB / Overlay export uses display-only context fill")
print("✓ Scientific label export still contains IGNORE=255")


In [ ]:
# ============================================================
# FINAL STABLE EXPORT — v1.3.1 AUDITED
#
# Scientific label map:
# - Google/UI screenshot residuals remain IGNORE=255
# - IGNORE pixels are excluded from every percentage denominator
# - artifact pixels NEVER count as pole/road/sidewalk/etc.
#
# Visual RGB / overlay:
# - ignored artifact holes are context-filled for readability only
# - visual fill does NOT affect statistics
# ============================================================

from pathlib import Path
from PIL import Image
import json
import numpy as np
import pandas as pd

STABLE_SHARE_VERSION = "vision_model_segmentation_taxonomy_stable_share_v1.3.1_audited_share_ready"

TEST_OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "stable_share_v1_3_1_audited_share_ready"
    / TEST_ID
)
TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Statistics: IGNORE excluded from denominator.
# ------------------------------------------------------------

valid_mask = final_v010 != IGNORE_ID
valid_pixel_count = max(int(valid_mask.sum()), 1)

rows_final = []

for class_id in sorted(CLASS_NAMES.keys()):
    class_name = CLASS_NAMES[class_id]

    pixel_count = int(np.sum(valid_mask & (final_v010 == class_id)))
    percentage = pixel_count / valid_pixel_count * 100.0

    rows_final.append({
        "class_id": class_id,
        "class_name": class_name,
        "pixels": pixel_count,
        "percentage": percentage
    })

stats_final = pd.DataFrame(rows_final).sort_values(
    ["percentage", "class_id"],
    ascending=[False, True]
).reset_index(drop=True)

unknown_pct = float(
    np.sum(valid_mask & (final_v010 == LABEL2ID["other_unknown"]))
    / valid_pixel_count
    * 100.0
)

ignore_pct_of_full_image = float(
    np.mean(final_v010 == IGNORE_ID) * 100.0
)

coverage_pct = 100.0 - unknown_pct

artifact_detected_pct = float(
    artifact_ignore_mask.mean() * 100.0
) if "artifact_ignore_mask" in globals() else 0.0

artifact_ignored_pixels = int(
    artifact_ignore_mask.sum()
) if "artifact_ignore_mask" in globals() else 0

# ------------------------------------------------------------
# HARD SCIENTIFIC QA
# ------------------------------------------------------------

if "artifact_ignore_mask" in globals():
    assert np.all(
        final_v010[artifact_ignore_mask] == IGNORE_ID
    )

artifact_pole_pixels = 0
if "artifact_ignore_mask" in globals() and "pole_fixture" in LABEL2ID:
    artifact_pole_pixels = int(np.sum(
        artifact_ignore_mask
        &
        (final_v010 == LABEL2ID["pole_fixture"])
    ))

assert artifact_pole_pixels == 0, (
    "Google/UI artifact contamination detected in pole_fixture."
)

# Full label-schema audit at export boundary.
declared_ids = set(CLASS_NAMES.keys())
observed_valid_ids = set(np.unique(final_v010[valid_mask]).astype(int).tolist())
assert observed_valid_ids.issubset(declared_ids), (
    f"Undeclared final labels: {sorted(observed_valid_ids - declared_ids)}"
)
assert NUM_CLASSES == TAXONOMY["num_classes"] == len(CVAT_LABELS) == 30

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

image.save(TEST_OUTPUT_DIR / f"{TEST_ID}_ORIGINAL.png")

# Scientific labels: 255 remains 255.
Image.fromarray(final_v010.astype(np.uint8)).save(
    TEST_OUTPUT_DIR / f"{TEST_ID}_LABEL_MAP_STABLE.png"
)

Image.fromarray(final_v010.astype(np.uint8)).save(
    TEST_OUTPUT_DIR / f"{TEST_ID}_PSEUDO_LABEL_STABLE.png"
)

# Display-clean outputs.
Image.fromarray(rgb_v010.astype(np.uint8)).save(
    TEST_OUTPUT_DIR / f"{TEST_ID}_RGB_CLEAN.png"
)

Image.fromarray(overlay_v010.astype(np.uint8)).save(
    TEST_OUTPUT_DIR / f"{TEST_ID}_OVERLAY_CLEAN.png"
)

# Optional display-only label map for auditability.
Image.fromarray(display_label_map.astype(np.uint8)).save(
    TEST_OUTPUT_DIR / f"{TEST_ID}_DISPLAY_CONTEXT_LABEL_MAP.png"
)

artifact_mask_export = (
    artifact_ignore_mask.astype(np.uint8) * 255
    if "artifact_ignore_mask" in globals()
    else np.zeros((H, W), dtype=np.uint8)
)

Image.fromarray(artifact_mask_export).save(
    TEST_OUTPUT_DIR / f"{TEST_ID}_STREETVIEW_ARTIFACT_IGNORE_MASK.png"
)

Image.fromarray(
    (final_v010 == IGNORE_ID).astype(np.uint8) * 255
).save(
    TEST_OUTPUT_DIR / f"{TEST_ID}_IGNORE_MASK.png"
)

stats_final.to_csv(
    TEST_OUTPUT_DIR / f"{TEST_ID}_STATISTICS_STABLE.csv",
    index=False,
    encoding="utf-8-sig"
)

diagnostics_final = {
    "test_id": TEST_ID,
    "stable_share_version": STABLE_SHARE_VERSION,
    "coverage_pct_excluding_ignore": coverage_pct,
    "unknown_pct_excluding_ignore": unknown_pct,
    "ignore_pct_of_full_image": ignore_pct_of_full_image,
    "streetview_artifact_ignored_pct": artifact_detected_pct,
    "streetview_artifact_ignored_pixels": artifact_ignored_pixels,
    "streetview_artifact_pixels_classified_as_pole": artifact_pole_pixels,
    "valid_pixels": valid_pixel_count,
    "full_image_pixels": int(final_v010.size),
    "declared_num_classes": NUM_CLASSES,
    "observed_valid_label_ids": sorted(observed_valid_ids),
    "scientific_rule": (
        "Google Street View screenshot/UI residual pixels are IGNORE=255 "
        "and excluded from taxonomy percentages."
    )
}

with open(
    TEST_OUTPUT_DIR / f"{TEST_ID}_DIAGNOSTICS_STABLE.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        diagnostics_final,
        f,
        ensure_ascii=False,
        indent=2
    )

manifest_final = {
    "stable_share_version": STABLE_SHARE_VERSION,
    "test_id": TEST_ID,
    "taxonomy_version": TAXONOMY.get("taxonomy_version", "unknown"),
    "ignore_index": IGNORE_ID,
    "declared_num_classes": NUM_CLASSES,
    "traffic_cone_barrel_id": LABEL2ID["traffic_cone_barrel"],
    "google_artifact_policy": "strict_ignore",
    "google_artifact_display_policy": "context_fill_visualization_only",
}

with open(
    TEST_OUTPUT_DIR / f"{TEST_ID}_RUN_MANIFEST_STABLE.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifest_final,
        f,
        ensure_ascii=False,
        indent=2
    )

print("========================================")
print("FINAL STABLE EXPORT — v1.3.1 AUDITED")
print("========================================")
print("Output:", TEST_OUTPUT_DIR)
print("Valid pixels:", valid_pixel_count)
print("IGNORE % of full image:", ignore_pct_of_full_image)
print("Google/UI ignored pixels:", artifact_ignored_pixels)
print("Google/UI pixels classified as pole:", artifact_pole_pixels)
print("✓ STRICT IGNORE + TAXONOMY SCHEMA QA PASSED")


In [ ]:
# ============================================================
# FINAL STABLE SUMMARY FIGURE — LEGEND VERSION
# No text overlap with color patches
# ============================================================

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

TITLE_SIZE = 28
LEGEND_FONT_SIZE = 18

# 用 2 欄最穩，不容易撞到
LEGEND_NCOLS = 2

FIG_W = 16
FIG_H = 13

SHOW_ZERO_CLASSES = False
MIN_PERCENT_TO_SHOW = 0.0

DISPLAY_NAME_MAP = {
    "other_unknown": "other unknown",
    "roadway": "roadway",
    "sidewalk": "sidewalk",
    "bike_lane": "bike lane",
    "curb_edge": "curb edge",
    "upper_building_facade": "upper facade",
    "ground_floor_solid_facade": "ground solid facade",
    "ground_floor_glazing": "ground glazing",
    "door_entrance": "door entrance",
    "signboard": "signboard",
    "awning_canopy": "awning canopy",
    "arcade_column": "arcade column",
    "arcade_soffit": "arcade soffit",
    "sidewalk_shed_scaffold": "scaffold",
    "stoop_stair": "stoop stair",
    "wall_ledge": "wall ledge",
    "fence_railing": "fence railing",
    "planter_container": "planter",
    "tree": "tree",
    "shrub_hedge": "shrub hedge",
    "ground_vegetation": "ground vegetation",
    "vertical_green_wall": "green wall",
    "bench_seating": "bench",
    "pole_fixture": "pole fixture",
    "traffic_sign_signal": "traffic sign/signal",
    "person": "person",
    "vehicle": "vehicle",
    "sky": "sky",
    "upper_building_glazing": "upper glazing",
    "traffic_cone_barrel": "traffic cone/barrel"
}


# ------------------------------------------------------------
# 1. Display-only label map
#    Scientific final_v010 keeps Google/UI pixels as IGNORE=255.
#    For the figure only, use the separate broad-context fill.
# ------------------------------------------------------------

display_label_map = (
    display_context_labels.copy()
    if "display_context_labels" in globals()
    else final_v010.copy()
)

if "IGNORE_ID" in globals():
    display_label_map[
        display_label_map == IGNORE_ID
    ] = UNKNOWN_ID


# ------------------------------------------------------------
# 2. Render semantic mask
# ------------------------------------------------------------

semantic_rgb = render_taxonomy(
    display_label_map
)


# ------------------------------------------------------------
# 3. Valid pixels for statistics
#    IGNORE excluded from denominator
# ------------------------------------------------------------

if "IGNORE_ID" in globals():
    valid_mask = (
        final_v010 != IGNORE_ID
    )
else:
    valid_mask = np.ones(
        final_v010.shape,
        dtype=bool
    )

valid_pixel_count = max(
    int(valid_mask.sum()),
    1
)


# ------------------------------------------------------------
# 4. Build summary table
# ------------------------------------------------------------

summary_rows = []

for class_id in sorted(CLASS_NAMES.keys()):

    class_name = CLASS_NAMES[class_id]

    pixel_count = int(
        np.sum(
            valid_mask &
            (final_v010 == class_id)
        )
    )

    percentage = (
        pixel_count / valid_pixel_count * 100
    )

    if not SHOW_ZERO_CLASSES and pixel_count == 0:
        continue

    if percentage < MIN_PERCENT_TO_SHOW:
        continue

    display_name = DISPLAY_NAME_MAP.get(
        class_name,
        class_name.replace("_", " ")
    )

    summary_rows.append({
        "class_id": class_id,
        "class_name": class_name,
        "display_name": display_name,
        "pixels": pixel_count,
        "percentage": percentage,
        "color": PALETTE[class_id]
    })

summary_df = pd.DataFrame(summary_rows)


# ------------------------------------------------------------
# 5. Sort:
#    main classes by descending percentage
#    other_unknown always last
# ------------------------------------------------------------

if len(summary_df) > 0:

    unknown_mask = (
        summary_df["class_name"] == "other_unknown"
    )

    df_main = summary_df[
        ~unknown_mask
    ].copy()

    df_unknown = summary_df[
        unknown_mask
    ].copy()

    df_main = df_main.sort_values(
        "percentage",
        ascending=False
    )

    summary_df_clean = pd.concat(
        [df_main, df_unknown],
        ignore_index=True
    )

else:
    summary_df_clean = summary_df.copy()


# ------------------------------------------------------------
# 6. Create legend handles
# ------------------------------------------------------------

handles = []
labels = []

for _, row in summary_df_clean.iterrows():

    color = np.array(row["color"]) / 255.0

    patch = mpatches.Patch(
        facecolor=color,
        edgecolor="none"
    )

    label = (
        f'{row["display_name"]} — '
        f'{row["percentage"]:.2f}%'
    )

    handles.append(patch)
    labels.append(label)


# ------------------------------------------------------------
# 7. Figure layout
# ------------------------------------------------------------

fig = plt.figure(
    figsize=(FIG_W, FIG_H),
    facecolor="white"
)

gs = fig.add_gridspec(
    2, 2,
    height_ratios=[3.1, 2.2],
    width_ratios=[1, 1],
    hspace=0.08,
    wspace=0.02
)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax_leg = fig.add_subplot(gs[1, :])

# Top images
ax1.imshow(image)
ax1.set_title(
    "Original SVI",
    fontsize=TITLE_SIZE,
    pad=12
)
ax1.axis("off")

ax2.imshow(semantic_rgb)
ax2.set_title(
    "Combined Semantic Mask",
    fontsize=TITLE_SIZE,
    pad=12
)
ax2.axis("off")

# Bottom legend panel
ax_leg.axis("off")

legend = ax_leg.legend(
    handles,
    labels,
    ncol=LEGEND_NCOLS,
    loc="center",
    frameon=False,
    fontsize=LEGEND_FONT_SIZE,
    handlelength=1.2,      # 色塊寬度
    handleheight=0.8,
    handletextpad=0.6,     # 色塊和文字距離
    columnspacing=1.4,     # 欄與欄距離
    labelspacing=1.0,      # 列與列距離
    borderpad=0.2
)

for text in legend.get_texts():
    text.set_ha("left")


plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------

if "TEST_OUTPUT_DIR" in globals():
    output_dir = Path(TEST_OUTPUT_DIR)
else:
    output_dir = Path.cwd() / "summary_outputs"
    output_dir.mkdir(parents=True, exist_ok=True)

summary_clean_figure_path = (
    output_dir / f"{TEST_ID}_SUMMARY_FIGURE_STABLE.png"
)

summary_clean_csv_path = (
    output_dir / f"{TEST_ID}_SUMMARY_PERCENTAGES_STABLE.csv"
)

fig.savefig(
    summary_clean_figure_path,
    dpi=220,
    bbox_inches="tight",
    facecolor="white"
)

summary_df_clean.to_csv(
    summary_clean_csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("✓ Final stable summary figure saved to:")
print(summary_clean_figure_path)

print()
print("✓ Final stable summary percentages CSV saved to:")
print(summary_clean_csv_path)

display(
    summary_df_clean[
        ["class_id", "class_name", "pixels", "percentage"]
    ]
)


## Share-version boundary

This notebook intentionally stops here.

### v1.3.1 audited invariants

- The taxonomy is declared once at initialization and contains **30 formal classes**.
- `traffic_cone_barrel` is formally declared as **ID 29**; it is not appended dynamically at runtime.
- Google Street View screenshot/UI residuals remain `IGNORE=255` and are excluded from measurement statistics.
- Park/metal fence recovery is class-specific and targets remaining `other_unknown` pixels only.
- Glass-dominant vertical building surfaces are vetoed from scaffold and restored to glazing/facade only when supported.
- Leaves, branches, and trunk are one output taxonomy class: `tree`.
- `other_unknown` is retained as a legitimate uncertainty class; there is no generic unknown-filling stage.
- Final QA asserts that all non-IGNORE output IDs belong to the declared taxonomy and that class percentages sum to 100% over valid pixels.

**This is the group-share single-image baseline.**
Do not append experimental unknown recovery, alternate thresholds, or batch experiments to this file.
